In [1]:
# Chunk 0 — runtime bootstrap (AutoDL / Colab / Kaggle one-click)
# 用法：
# 1) 通常只改 PLATFORM_OVERRIDE 和 MODEL paths / data paths
# 2) PLATFORM_OVERRIDE = "auto" 时会自动识别运行环境
# 3) Colab 下如果你已经把数据或模型放到 Drive，本格会优先直接读取本地目录
# 4) 如果本地目录不存在，Colab 会尝试用 kagglehub 下载（需要你先完成 Kaggle 认证）

import os
import sys
from pathlib import Path

PLATFORM_OVERRIDE = "autodl"   # "auto" | "autodl" | "colab" | "kaggle"

# ---------- Colab ----------
COLAB_USE_DRIVE = True
COLAB_DRIVE_ROOT = "/content/drive/MyDrive/nyu322"

# 如果你已经手动把比赛 csv 放到 Drive，可填这个目录；否则保持 None，用 kagglehub 下载
COLAB_DATA_DIR = None

# 如果模型已经放到 Drive，可直接填目录；否则保持 None，用 kagglehub / snapshot_download 下载
COLAB_MODEL_DIR_0P5B = os.path.join(COLAB_DRIVE_ROOT, "models/Qwen/Qwen2.5-Coder-0.5B-Instruct")
COLAB_MODEL_DIR_1P5B = os.path.join(COLAB_DRIVE_ROOT, "models/Qwen/Qwen2.5-Coder-1.5B-Instruct")

# ---------- AutoDL ----------
AUTODL_DATA_DIR = os.environ.get("SVG_DATA_DIR", "/root/autodl-tmp")
AUTODL_MODEL_DIR_0P5B = os.environ.get("SVG_MODEL_DIR_0P5B", "/root/autodl-tmp/models/Qwen/Qwen2.5-Coder-0.5B-Instruct")
AUTODL_MODEL_DIR_1P5B = os.environ.get("SVG_MODEL_DIR_1P5B", "/root/autodl-tmp/models/Qwen/Qwen2.5-Coder-1.5B-Instruct")
AUTODL_WORK_ROOT = os.environ.get("SVG_WORK_ROOT", "/root/autodl-tmp/working/nyu322_pipeline1")
AUTODL_OUTPUT_ROOT = os.environ.get("SVG_OUTPUT_ROOT", "/root/autodl-tmp/output/nyu322_pipeline1")

# ---------- Kaggle ----------
KAGGLE_COMPETITION_SLUG = "dl-spring-2026-svg-generation"
KAGGLE_MODEL_DIR_0P5B = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/0.5b-instruct/1"
KAGGLE_MODEL_DIR_1P5B = "/kaggle/input/models/qwen-lm/qwen2.5/transformers/1.5b-instruct/1"

# ---------- Shared repo ids ----------
KAGGLE_MODEL_HANDLE_0P5B = "qwen-lm/qwen2.5/Transformers/0.5b-instruct/1"
KAGGLE_MODEL_HANDLE_1P5B = "qwen-lm/qwen2.5/Transformers/1.5b-instruct/1"
HF_REPO_0P5B = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
HF_REPO_1P5B = "Qwen/Qwen2.5-Coder-1.5B-Instruct"

def _path_has_competition_csvs(root: str) -> bool:
    if not root:
        return False
    return all(os.path.exists(os.path.join(root, name)) for name in ["train.csv", "test.csv", "sample_submission.csv"])

def _ensure_dir(path: str) -> str:
    os.makedirs(path, exist_ok=True)
    return path

def detect_runtime() -> str:
    if PLATFORM_OVERRIDE != "auto":
        return PLATFORM_OVERRIDE.lower().strip()

    if os.path.exists("/kaggle/working"):
        return "kaggle"

    try:
        import google.colab  # type: ignore
        return "colab"
    except Exception:
        pass

    if os.path.exists("/root/autodl-tmp"):
        return "autodl"

    raise RuntimeError(
        "Could not detect runtime automatically. "
        "Please set PLATFORM_OVERRIDE to 'autodl', 'colab', or 'kaggle'."
    )

def maybe_mount_colab_drive() -> None:
    if detect_runtime() != "colab" or not COLAB_USE_DRIVE:
        return
    try:
        from google.colab import drive  # type: ignore
        if not os.path.exists("/content/drive"):
            drive.mount("/content/drive")
    except Exception as e:
        print("[WARN] Could not mount Google Drive:", repr(e))

def _import_kagglehub():
    try:
        import kagglehub
        return kagglehub
    except Exception:
        raise RuntimeError(
            "kagglehub is required for automatic Colab downloads. "
            "Install it with: !pip install -q kagglehub"
        )

def resolve_competition_data_dir(runtime: str) -> str:
    if runtime == "kaggle":
        return f"/kaggle/input/competitions/{KAGGLE_COMPETITION_SLUG}"

    if runtime == "autodl":
        if not _path_has_competition_csvs(AUTODL_DATA_DIR):
            raise FileNotFoundError(
                f"AutoDL data dir missing train/test/sample_submission csvs: {AUTODL_DATA_DIR}"
            )
        return AUTODL_DATA_DIR

    if runtime == "colab":
        maybe_mount_colab_drive()
        if COLAB_DATA_DIR and _path_has_competition_csvs(COLAB_DATA_DIR):
            return COLAB_DATA_DIR

        kagglehub = _import_kagglehub()
        return kagglehub.competition_download(KAGGLE_COMPETITION_SLUG)

    raise ValueError(f"Unsupported runtime: {runtime}")

def resolve_model_dir(runtime: str, model_size: str) -> str:
    model_size = model_size.lower().strip()
    if model_size not in {"0.5b", "1.5b"}:
        raise ValueError("model_size must be '0.5b' or '1.5b'")

    if runtime == "kaggle":
        return KAGGLE_MODEL_DIR_0P5B if model_size == "0.5b" else KAGGLE_MODEL_DIR_1P5B

    if runtime == "autodl":
        model_dir = AUTODL_MODEL_DIR_0P5B if model_size == "0.5b" else AUTODL_MODEL_DIR_1P5B
        if not os.path.exists(model_dir):
            raise FileNotFoundError(f"AutoDL model dir not found: {model_dir}")
        return model_dir

    if runtime == "colab":
        maybe_mount_colab_drive()
        manual_dir = COLAB_MODEL_DIR_0P5B if model_size == "0.5b" else COLAB_MODEL_DIR_1P5B
        if manual_dir and os.path.exists(manual_dir):
            return manual_dir

        kagglehub = _import_kagglehub()
        handle = KAGGLE_MODEL_HANDLE_0P5B if model_size == "0.5b" else KAGGLE_MODEL_HANDLE_1P5B
        return kagglehub.model_download(handle)

    raise ValueError(f"Unsupported runtime: {runtime}")

def resolve_work_roots(runtime: str) -> dict:
    if runtime == "kaggle":
        work_root = "/kaggle/working/nyu322_pipeline"
        output_root = "/kaggle/working/nyu322_outputs"
    elif runtime == "autodl":
        work_root = AUTODL_WORK_ROOT
        output_root = AUTODL_OUTPUT_ROOT
    elif runtime == "colab":
        maybe_mount_colab_drive()
        base_root = COLAB_DRIVE_ROOT if COLAB_USE_DRIVE else "/content/nyu322"
        work_root = os.path.join(base_root, "working", "nyu322_pipeline")
        output_root = os.path.join(base_root, "output", "nyu322_pipeline")
    else:
        raise ValueError(f"Unsupported runtime: {runtime}")

    _ensure_dir(work_root)
    _ensure_dir(output_root)
    return {"work_root": work_root, "output_root": output_root}

RUNTIME_PLATFORM = detect_runtime()
RUNTIME_DATA_DIR = resolve_competition_data_dir(RUNTIME_PLATFORM)
RUNTIME_ROOTS = resolve_work_roots(RUNTIME_PLATFORM)

print("Detected runtime :", RUNTIME_PLATFORM)
print("Data dir         :", RUNTIME_DATA_DIR)
print("Work root        :", RUNTIME_ROOTS["work_root"])
print("Output root      :", RUNTIME_ROOTS["output_root"])


Detected runtime : autodl
Data dir         : /root/autodl-tmp
Work root        : /root/autodl-tmp/working/nyu322_pipeline1
Output root      : /root/autodl-tmp/output/nyu322_pipeline1


In [2]:
# Chunk 0.5 — optional one-time model download helper
# 默认不执行。
# 适合 Colab 第一次把 HF 模型下载到 Drive；下载完成后把 RUN_MODEL_DOWNLOAD 重新关掉。

RUN_MODEL_DOWNLOAD = False
DOWNLOAD_MODEL_SIZE = "1.5b"   # "0.5b" | "1.5b"

if RUN_MODEL_DOWNLOAD:
    from huggingface_hub import snapshot_download

    if RUNTIME_PLATFORM != "colab":
        raise RuntimeError("This helper is mainly intended for Colab.")
    maybe_mount_colab_drive()

    if DOWNLOAD_MODEL_SIZE == "0.5b":
        target_dir = COLAB_MODEL_DIR_0P5B
        repo_id = HF_REPO_0P5B
    elif DOWNLOAD_MODEL_SIZE == "1.5b":
        target_dir = COLAB_MODEL_DIR_1P5B
        repo_id = HF_REPO_1P5B
    else:
        raise ValueError("DOWNLOAD_MODEL_SIZE must be '0.5b' or '1.5b'")

    os.makedirs(target_dir, exist_ok=True)
    snapshot_download(
        repo_id=repo_id,
        local_dir=target_dir,
        local_dir_use_symlinks=False,
    )
    print("Downloaded model to:", target_dir)
else:
    print("RUN_MODEL_DOWNLOAD = False -> skip")


RUN_MODEL_DOWNLOAD = False -> skip


In [3]:
# Chunk 0.6 — optional quick sanity checks
def _show_dir_head(path: str, n: int = 10):
    if not os.path.exists(path):
        print("[missing]", path)
        return
    print(path)
    for name in sorted(os.listdir(path))[:n]:
        print(" -", name)

print("Runtime platform:", RUNTIME_PLATFORM)
_show_dir_head(RUNTIME_DATA_DIR)


Runtime platform: autodl
/root/autodl-tmp
 - .Trash-0
 - .autodl
 - .ipynb_checkpoints
 - =0.46.1
 - autodl-tmp
 - models
 - notebook_svg_ab_baseline_hybrid100_fixed.ipynb
 - notebook_svg_ab_canon_only_filtered.ipynb
 - notebook_svg_ab_curriculum_canon_emh.ipynb
 - notebook_svg_ab_curriculum_only_emh.ipynb


In [4]:
# Chunk 0.7 — deprecated Kaggle import cell removed
print("Deprecated Kaggle import cell removed. Use Chunk 0 instead.")


Deprecated Kaggle import cell removed. Use Chunk 0 instead.


# NYU322 SVG Kaggle — end-to-end direct LoRA run + submission

## Before you run

In Kaggle, make sure you have attached:
1. the **competition dataset** so `train.csv / test.csv / sample_submission.csv` are visible
2. the **base model dataset** such as `Qwen2.5-Coder-0.5B-Instruct`
3. a **GPU notebook session**

This notebook now follows the **official competition SVG tag policy** by default:
- max 16000 characters
- max 256 `<path>` elements
- allowed tags only: `svg, g, path, rect, circle, ellipse, line, polyline, polygon, defs, use, symbol, clipPath, mask, linearGradient, radialGradient, stop, text, tspan, title, desc, style, pattern, marker, filter`

Notes:
- default config uses `TAG_MODE = "official"`
- `TAG_MODE = "simple"` is still available only for ablation / debugging
- scripts, external references, animation, and other clearly unsafe tags are still blocked


In [5]:
# Chunk X — deprecated hardcoded model download cell removed
print("Deprecated hardcoded download cell removed. Use Chunk 0.5 instead if needed.")


Deprecated hardcoded download cell removed. Use Chunk 0.5 instead if needed.


In [6]:

# Chunk X-0 — install local metric deps (run once if needed)
# 这里只强依赖 cairosvg / zss。
# scikit-image 不是硬依赖：如果没有装，后面的 proxy metric 会自动走 fallback 实现。
# 如果你明确想用 skimage 版本，再手动安装:
# !pip -q install scikit-image

try:
    import cairosvg  # noqa
    import zss       # noqa
except Exception:
    !pip -q install cairosvg zss

try:
    from skimage.metrics import structural_similarity as _skimage_ssim  # noqa
    from skimage.feature import canny as _skimage_canny  # noqa
    print("scikit-image detected -> proxy metric will use skimage backend.")
except Exception:
    print("scikit-image not found -> proxy metric will use numpy fallback backend.")


scikit-image detected -> proxy metric will use skimage backend.


In [7]:

# Chunk 1 — environment check
import os
import sys
import gc
import math
import random
import re
import warnings
from dataclasses import dataclass
from typing import Dict, List, Optional, Sequence, Tuple
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("CUDA device count:", torch.cuda.device_count())
    print("Current device:", torch.cuda.current_device())
    print("Device name:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())
else:
    print("GPU is not enabled. For Kaggle training, turn GPU on in notebook settings.")


Python: 3.12.3 | packaged by Anaconda, Inc. | (main, May  6 2024, 19:46:43) [GCC 11.2.0]
Torch: 2.10.0+cu128
CUDA available: True
CUDA device count: 1
Current device: 0
Device name: NVIDIA GeForce RTX 5090
BF16 supported: True


In [8]:

# Chunk 2 — ML / LoRA imports
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from sklearn.model_selection import train_test_split, StratifiedKFold, KFold

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    PeftModel,
    get_peft_model,
    prepare_model_for_kbit_training,
)



## Config

The config below is organized so you can switch between:
- **debug mode**: small sample, fast sanity checks
- **full mode**: train on the whole competition set

Recommended workflow:
1. run once with `DEBUG_MODE = True`
2. confirm cleaning, training, preview generation, and submission creation all work
3. switch to `DEBUG_MODE = False`


In [9]:
# Chunk 3 — configuration / experiment switches
# 这一格是总控制台：
# 1) 先切平台（auto / autodl / kaggle / colab）
# 2) 再切模型大小（0.5b / 1.5b）
# 3) 再切 retrieval 数据源（full / filtered / split）
# 4) 再切 submission 模式（retrieval_only / hybrid / generation_first）
# 5) 这版保留单模型直跑，但把平台路径层抽成真正独立的一层
#
# 当前默认：
# - PLATFORM = "auto"（推荐）
# - MAX_SEQ_LEN = 2048
# - MAX_NEW_TOKENS = 896
# - direct run（no OOF / no stage2）
# - filtered 单模型训练 + train/valid validation

import os
import random
import numpy as np

def seed_everything(seed: int = 42) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        try:
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
        except Exception:
            pass
    except Exception:
        pass

@dataclass
class CFG:
    # =========================
    # Experiment control plane
    # =========================
    PLATFORM: str = "auto"      # "auto" | "autodl" | "kaggle" | "colab"
    MODEL_SIZE: str = "1.5b"    # "0.5b" | "1.5b"

    RETRIEVAL_SOURCE: str = "full"   # "full" | "filtered" | "split"
    SUBMISSION_MODE: str = "hybrid"  # "retrieval_only" | "hybrid" | "generation_first"

    TAG_MODE: str = "official"       # "simple" | "official"
    SANITIZER_MODE: str = "recover"  # "strict" | "recover"

    QUICK_HOLDOUT_SIZE: float = 0.20
    QUICK_EVAL_MAX_ROWS: Optional[int] = 300
    RUN_QUICK_EXPERIMENTS: bool = False

    RUN_OOF_CONFIRM: bool = False
    OOF_NUM_FOLDS: int = 3
    OOF_TOP_N: int = 2
    OOF_MAX_ROWS_PER_FOLD: Optional[int] = 300
    MIN_VALID_RATE_FOR_SELECTION: float = 0.995

    FINAL_RUN_NAME: Optional[str] = None
    AUTO_PICK_BEST_RUN: bool = True

    # preset / training plan
    EXPERIMENT_PRESET: str = "baseline_hybrid100"
    TRAIN_PLAN: str = "single"              # "single" | "curriculum"

    # canonicalization / curriculum flags (kept for compatibility)
    ENABLE_CANONICALIZATION: bool = False
    CANON_DECIMALS: int = 2
    CANON_SORT_ATTRS: bool = True
    CANON_NORMALIZE_COLORS: bool = True
    CANON_REMOVE_REDUNDANT_ATTRS: bool = True
    CANON_COLLAPSE_GROUPS: bool = False
    CANON_NORMALIZE_STYLE: bool = False

    CURRICULUM_STAGEA_SOURCE: str = "curriculum_stage_a"
    CURRICULUM_STAGEB_SOURCE: str = "curriculum_stage_b"
    CURRICULUM_STAGEC_SOURCE: str = "hard_focus"
    CURRICULUM_STAGEA_EPOCHS: float = 0.6
    CURRICULUM_STAGEB_EPOCHS: float = 0.6
    CURRICULUM_STAGEC_EPOCHS: float = 0.3
    CURRICULUM_STAGEC_FAMILIES: Tuple[str, ...] = (
        "container_frame",
        "device_object",
        "person_organic",
        "composite_relation",
    )
    INIT_ADAPTER_DIR: Optional[str] = None
    RUN_TAG: str = "v1"
    MODEL_TAG: str = ""

    REUSE_EXISTING_TRAINED_ADAPTERS: bool = True
    INITIAL_LORA_SOURCES: Tuple[str, ...] = ("filtered",)
    INITIAL_LORA_EPOCH: float = 1.0
    BONUS_LORA_EPOCH: float = 1.0

    USE_TRAIN_VALIDATION: bool = True
    TRAIN_VALID_SIZE: float = 0.08
    TRAIN_VALID_MAX_ROWS: Optional[int] = None
    TRAIN_VALID_MIN_ROWS: int = 128

    RUN_PROXY_METRIC: bool = True
    PROXY_EVAL_MODE: str = "both"
    PROXY_MAX_ROWS: Optional[int] = 300

    DIRECT_RETRIEVAL_THRESHOLD: float = 1.00
    ENABLE_PREVIEW_GENERATION: bool = True
    PREVIEW_MAX_ROWS: int = 8
    SAVE_DEBUG_CSV: bool = True

    DO_TRAIN: bool = True
    LOAD_EXISTING_ADAPTER: bool = False
    LORA_TRAIN_SOURCE: str = "filtered"
    DO_INFERENCE: bool = True

    STRICT_ADAPTER_MATCH: bool = True
    AUTO_ARCHIVE_MISMATCHED_ADAPTER: bool = True
    ALLOW_BASE_ONLY_INFERENCE_IF_ADAPTER_MISMATCH: bool = True
    MERGE_ADAPTER_FOR_INFERENCE: bool = False

    DEBUG_MODE: bool = False
    SEED: int = 42
    DEBUG_TRAIN_SAMPLES: Optional[int] = None
    DEBUG_QUICK_HOLDOUT_SAMPLES: Optional[int] = None
    DEBUG_VALID_SAMPLES: Optional[int] = None

    # 评分里 C 只占 3%，因此训练清洗不要过早把复杂样本压得太狠；
    # 默认仍保守，但你可以围绕这些阈值做“中复杂度扩容”实验。
    MAX_SVG_CHARS: int = 16000
    MAX_PATHS: int = 256
    MAX_TEXT_CHARS: int = 200
    MAX_STYLE_CHARS: int = 400

    FILTER_MAX_SVG_LEN: int = 4200
    FILTER_MAX_PATHS: int = 72
    FILTER_MIN_PROMPT_WORDS: int = 4
    FILTER_MAX_PROMPT_WORDS: int = 36
    FILTER_MAX_COMMANDS: int = 180
    FILTER_MAX_CONTROL_POINTS: int = 160
    FILTER_MAX_COMMAND_DIVERSITY: int = 8
    FILTER_MAX_PRIMITIVE_TYPES: int = 4
    FILTER_MAX_MEAN_PATH_D_LEN: int = 900
    FILTER_MAX_MAX_PATH_D_LEN: int = 1600
    FILTER_ALLOW_COMPLEX_EFFECTS: bool = False
    FILTER_ALLOW_TEXT: bool = False
    FILTER_REQUIRE_VISIBLE: bool = True

    MAX_SEQ_LEN: int = 2048
    MAX_NEW_TOKENS: int = 896
    USE_4BIT: bool = False
    BF16: bool = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    GRADIENT_CHECKPOINTING: bool = True
    TRAIN_OPTIM: str = "adamw_torch"

    LORA_R: int = 16
    LORA_ALPHA: int = 32
    LORA_DROPOUT: float = 0.05
    TARGET_MODULES: Tuple[str, ...] = (
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    )

    NUM_EPOCHS: float = 1.0
    TRAIN_BATCH_SIZE: int = 1
    EVAL_BATCH_SIZE: int = 1
    GRAD_ACCUM_STEPS: int = 4
    LEARNING_RATE: float = 2e-4
    WEIGHT_DECAY: float = 0.01
    WARMUP_STEPS: int = 50
    LOGGING_STEPS: int = 20
    SAVE_STEPS: int = 250
    EVAL_STEPS: int = 250

    TEMPERATURE: float = 0.2
    TOP_P: float = 0.9
    DO_SAMPLE: bool = False
    NUM_BEAMS: int = 1
    REPETITION_PENALTY: float = 1.10

    TRAIN_CSV: str = ""
    TEST_CSV: str = ""
    SAMPLE_SUBMISSION_CSV: str = ""
    BASE_MODEL_PATH: str = ""
    OUTPUT_DIR: str = ""
    ADAPTER_DIR: str = ""
    SUBMISSION_PATH: str = ""

    def __post_init__(self):
        platform = self.PLATFORM.lower().strip()
        model_size = self.MODEL_SIZE.lower().strip()

        if platform == "auto":
            platform = RUNTIME_PLATFORM

        if platform not in {"autodl", "kaggle", "colab"}:
            raise ValueError("CFG.PLATFORM must be 'auto', 'autodl', 'kaggle', or 'colab'")
        if model_size not in {"0.5b", "1.5b"}:
            raise ValueError("CFG.MODEL_SIZE must be '0.5b' or '1.5b'")
        if self.RETRIEVAL_SOURCE not in {"full", "filtered", "split"}:
            raise ValueError("CFG.RETRIEVAL_SOURCE must be 'full', 'filtered', or 'split'")
        if self.SUBMISSION_MODE not in {"retrieval_only", "hybrid", "generation_first"}:
            raise ValueError("CFG.SUBMISSION_MODE must be 'retrieval_only', 'hybrid', or 'generation_first'")
        if self.LORA_TRAIN_SOURCE not in {"filtered", "full", "split"}:
            raise ValueError("CFG.LORA_TRAIN_SOURCE must be 'filtered', 'full', or 'split'")
        if self.TAG_MODE not in {"simple", "official"}:
            raise ValueError("CFG.TAG_MODE must be 'simple' or 'official'")
        if self.SANITIZER_MODE not in {"strict", "recover"}:
            raise ValueError("CFG.SANITIZER_MODE must be 'strict' or 'recover'")
        if not (0 < self.QUICK_HOLDOUT_SIZE < 0.5):
            raise ValueError("CFG.QUICK_HOLDOUT_SIZE should be between 0 and 0.5.")
        if not (0 < self.TRAIN_VALID_SIZE < 0.5):
            raise ValueError("CFG.TRAIN_VALID_SIZE should be between 0 and 0.5.")
        if self.FILTER_MIN_PROMPT_WORDS >= self.FILTER_MAX_PROMPT_WORDS:
            raise ValueError("FILTER_MIN_PROMPT_WORDS must be < FILTER_MAX_PROMPT_WORDS")

        model_tag = "0p5b" if model_size == "0.5b" else "1p5b"
        self.MODEL_TAG = model_tag

        data_dir = resolve_competition_data_dir(platform)
        model_dir = resolve_model_dir(platform, model_size)
        roots = resolve_work_roots(platform)

        self.TRAIN_CSV = os.path.join(data_dir, "train.csv")
        self.TEST_CSV = os.path.join(data_dir, "test.csv")
        self.SAMPLE_SUBMISSION_CSV = os.path.join(data_dir, "sample_submission.csv")
        self.BASE_MODEL_PATH = model_dir

        self.OUTPUT_DIR = roots["work_root"]
        adapter_root = os.path.join(self.OUTPUT_DIR, "adapters")
        submission_root = roots["output_root"]
        os.makedirs(adapter_root, exist_ok=True)
        os.makedirs(submission_root, exist_ok=True)

        self.ADAPTER_DIR = os.path.join(adapter_root, f"svg_lora_adapter_{model_tag}")
        self.SUBMISSION_PATH = os.path.join(submission_root, f"submission_{platform}_{model_tag}.csv")

CFG = CFG()

print("Platform                 :", CFG.PLATFORM, "-> active:", RUNTIME_PLATFORM if CFG.PLATFORM == "auto" else CFG.PLATFORM)
print("Model size               :", CFG.MODEL_SIZE)
print("Retrieval source         :", CFG.RETRIEVAL_SOURCE)
print("Submission mode          :", CFG.SUBMISSION_MODE)
print("Tag mode                 :", CFG.TAG_MODE)
print("Sanitizer mode           :", CFG.SANITIZER_MODE)
print("Experiment preset        :", CFG.EXPERIMENT_PRESET)
print("Train plan               :", CFG.TRAIN_PLAN)
print("Quick holdout size       :", CFG.QUICK_HOLDOUT_SIZE)
print("Use train validation     :", CFG.USE_TRAIN_VALIDATION)
print("Train validation size    :", CFG.TRAIN_VALID_SIZE)
print("MAX_SEQ_LEN              :", CFG.MAX_SEQ_LEN)
print("MAX_NEW_TOKENS           :", CFG.MAX_NEW_TOKENS)
print("Train csv                :", CFG.TRAIN_CSV)
print("Test csv                 :", CFG.TEST_CSV)
print("Base model path          :", CFG.BASE_MODEL_PATH)
print("Output dir               :", CFG.OUTPUT_DIR)
print("Adapter dir              :", CFG.ADAPTER_DIR)
print("Submission path          :", CFG.SUBMISSION_PATH)

seed_everything(CFG.SEED)


Platform                 : auto -> active: autodl
Model size               : 1.5b
Retrieval source         : full
Submission mode          : hybrid
Tag mode                 : official
Sanitizer mode           : recover
Experiment preset        : baseline_hybrid100
Train plan               : single
Quick holdout size       : 0.2
Use train validation     : True
Train validation size    : 0.08
MAX_SEQ_LEN              : 2048
MAX_NEW_TOKENS           : 896
Train csv                : /root/autodl-tmp/train.csv
Test csv                 : /root/autodl-tmp/test.csv
Base model path          : /root/autodl-tmp/models/Qwen/Qwen2.5-Coder-1.5B-Instruct
Output dir               : /root/autodl-tmp/working/nyu322_pipeline1
Adapter dir              : /root/autodl-tmp/working/nyu322_pipeline1/adapters/svg_lora_adapter_1p5b
Submission path          : /root/autodl-tmp/output/nyu322_pipeline1/submission_autodl_1p5b.csv


In [10]:

# Chunk 3.1 — experiment presets for A/B on V / S
# baseline notebook: no curriculum
# 可选：
# - "baseline_hybrid100"
# - "canon_only_filtered"
# - "medium_only"

EXPERIMENT_PRESET = "baseline_hybrid100"

def apply_experiment_preset(cfg, preset: str) -> None:
    preset = str(preset).strip().lower()

    cfg.EXPERIMENT_PRESET = preset
    cfg.TRAIN_PLAN = "single"
    cfg.ENABLE_CANONICALIZATION = False
    cfg.CANON_DECIMALS = 2
    cfg.CANON_SORT_ATTRS = True
    cfg.CANON_NORMALIZE_COLORS = True
    cfg.CANON_REMOVE_REDUNDANT_ATTRS = True
    cfg.CANON_COLLAPSE_GROUPS = False
    cfg.CANON_NORMALIZE_STYLE = False

    cfg.INIT_ADAPTER_DIR = None
    cfg.RUN_TAG = "v1"

    cfg.SUBMISSION_MODE = "hybrid"
    cfg.DIRECT_RETRIEVAL_THRESHOLD = 1.00
    cfg.RETRIEVAL_SOURCE = "full"
    cfg.MAX_SEQ_LEN = 2048
    cfg.MAX_NEW_TOKENS = 896
    cfg.LOAD_EXISTING_ADAPTER = False
    cfg.DO_TRAIN = True
    cfg.DO_INFERENCE = True
    cfg.ENABLE_PREVIEW_GENERATION = True

    if preset == "baseline_hybrid100":
        cfg.LORA_TRAIN_SOURCE = "filtered"
        cfg.INITIAL_LORA_SOURCES = ("filtered",)
        cfg.INITIAL_LORA_EPOCH = 1.0

    elif preset == "canon_only_filtered":
        cfg.ENABLE_CANONICALIZATION = True
        cfg.LORA_TRAIN_SOURCE = "filtered"
        cfg.INITIAL_LORA_SOURCES = ("filtered",)
        cfg.INITIAL_LORA_EPOCH = 1.0

    elif preset == "medium_only":
        cfg.LORA_TRAIN_SOURCE = "filtered_medium"
        cfg.INITIAL_LORA_SOURCES = ("filtered_medium",)
        cfg.INITIAL_LORA_EPOCH = 1.0

    else:
        raise ValueError(f"Unknown EXPERIMENT_PRESET: {preset}")

apply_experiment_preset(CFG, EXPERIMENT_PRESET)

print("Experiment preset       :", CFG.EXPERIMENT_PRESET)
print("Train plan              :", CFG.TRAIN_PLAN)
print("Enable canonicalization :", CFG.ENABLE_CANONICALIZATION)
print("LoRA train source       :", CFG.LORA_TRAIN_SOURCE)
print("Retrieval source        :", CFG.RETRIEVAL_SOURCE)
print("Initial LoRA epoch      :", CFG.INITIAL_LORA_EPOCH)
print("Init adapter dir        :", CFG.INIT_ADAPTER_DIR)


Experiment preset       : baseline_hybrid100
Train plan              : single
Enable canonicalization : False
LoRA train source       : filtered
Retrieval source        : full
Initial LoRA epoch      : 1.0
Init adapter dir        : None


## A/B preset in this file

This file defaults to `baseline_hybrid100`.

Recommended submission tag:
- preset: `baseline_hybrid100`
- retrieval: `hybrid100/full`
- seq/new tokens: `2048/896`

### Direct-run updates in this version

This version keeps the **single-run direct path**, but adds three practical upgrades:

- **2048 / 896 is fully consistent** across training, preview, and submission overrides
- **training-side SVG cleaning** is used before feature extraction and filtering
- **complexity-aware filtered training** now uses not only `svg_len` / `path_count`, but also
  command count, command diversity, control-point proxy, primitive diversity, and long-path indicators

It also restores a small **train/valid split** for the chosen LoRA source so you can monitor `eval_loss`
without bringing back the old multi-candidate compare loop.

### Control panel usage

推荐直跑顺序：

1. `PLATFORM` 先切成你当前环境：`"autodl"`、`"kaggle"` 或 `"colab"`
2. `MODEL_SIZE` 第一阶段先确认你要跑 `"0.5b"` 还是 `"1.5b"`
3. `LORA_TRAIN_SOURCE` 默认继续用 `"filtered"`
4. `USE_TRAIN_VALIDATION = True`，让训练时保留一小块 validation
5. 直接训练单个 `filtered @ 2048 @ 1.0` adapter，然后出 preview / submission

几个重要说明：

- 这版**不恢复候选比较 / OOF 选模**
- `filtered_train_df` 现在是**复杂度感知过滤**，不是只看长度和 path 数
- `RETRIEVAL_SOURCE = "full"` 仍然适合最终 submission
- `SUBMISSION_MODE = "hybrid"` 仍然是默认主线路

In [11]:

# Chunk 4 — constants and helpers
from typing import Any
from contextlib import contextmanager
from dataclasses import fields as dataclass_fields

SIMPLE_ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
}

OFFICIAL_ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse", "line", "polyline", "polygon",
    "defs", "use", "symbol", "clipPath", "mask",
    "linearGradient", "radialGradient", "stop",
    "text", "tspan", "title", "desc",
    "style", "pattern", "marker", "filter",
}

ALWAYS_BLOCKED_TAGS = {
    "script", "foreignObject", "animate", "animateMotion", "animateTransform", "set",
    "image", "iframe", "audio", "video", "canvas", "embed", "object",
}

EMPTY_SVG = '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256"></svg>'
ADAPTER_META_FILENAME = "adapter_meta.json"

_whitespace_re = re.compile(r"\s+")
_NUMERIC_ATTRS_SINGLE = {
    "width", "height", "x", "y", "x1", "y1", "x2", "y2",
    "cx", "cy", "r", "rx", "ry",
    "stroke-width", "opacity", "fill-opacity",
}
_NUMERIC_ATTRS_MULTI = {
    "viewBox", "d", "points", "transform", "stroke-dasharray",
}

_num_re = re.compile(r"-?\d+(?:\.\d+)?")

EXTERNAL_REF_PATTERNS = (
    "http://", "https://", "//", "data:", "javascript:", "file:", "ftp:", "mailto:"
)

TEXT_LIKE_TAGS = {"text", "tspan", "title", "desc", "style"}

def allowed_tags_for_mode(mode: Optional[str] = None) -> set:
    mode = (mode or CFG.TAG_MODE).lower().strip()
    if mode == "simple":
        return set(SIMPLE_ALLOWED_TAGS)
    if mode == "official":
        return set(OFFICIAL_ALLOWED_TAGS)
    raise ValueError(f"Unknown tag mode: {mode}")

def build_system_prompt(tag_mode: Optional[str] = None) -> str:
    mode = (tag_mode or CFG.TAG_MODE).lower().strip()
    if mode == "simple":
        allowed = ", ".join(sorted(SIMPLE_ALLOWED_TAGS))
        return (
            "You generate exactly one compact, valid SVG icon and nothing else. "
            "Output raw SVG only, not markdown, not code fences, not explanations. "
            "The output must start with <svg and end with </svg>. "
            "Canvas must be width='256' height='256' viewBox='0 0 256 256'. "
            f"Allowed tags only: {allowed}. "
            "Do not use comments, scripts, external references, animation, gradients, masks, clipPaths, defs, or text."
        )

    allowed = ", ".join(sorted(OFFICIAL_ALLOWED_TAGS))
    return (
        "You generate exactly one compact, valid SVG and nothing else. "
        "Output raw SVG only, not markdown, not code fences, not explanations. "
        "The output must start with <svg and end with </svg>. "
        "Canvas must be width='256' height='256' viewBox='0 0 256 256'. "
        f"Allowed tags only: {allowed}. "
        "You may use defs, gradients, text, tspan, clipPath, mask, use, pattern, marker, filter, title, and desc "
        "if they are necessary and remain valid. Prefer compact markup. "
        "Never use scripts, external URLs, animation, foreignObject, or event handlers."
    )

def current_allowed_tags() -> set:
    return allowed_tags_for_mode(CFG.TAG_MODE)

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)

def local_name(tag: str) -> str:
    return tag.split("}", 1)[1] if "}" in tag else tag

def minify_svg(svg: str) -> str:
    svg = svg.replace("\n", " ").replace("\t", " ").replace("\r", " ")
    svg = _whitespace_re.sub(" ", svg).strip()
    return svg.replace("> <", "><")

def trim_after_svg(text: str) -> str:
    if not isinstance(text, str):
        return text
    lower = text.lower()
    end_tag = "</svg>"
    pos = lower.find(end_tag)
    if pos != -1:
        return text[:pos + len(end_tag)]
    return text

def trim_to_last_complete_tag(text: str) -> str:
    if not isinstance(text, str):
        return ""
    last_gt = text.rfind(">")
    if last_gt == -1:
        return text
    return text[: last_gt + 1]

def extract_svg_candidate(text: str) -> Optional[str]:
    if not isinstance(text, str) or len(text.strip()) == 0:
        return None

    text = (
        text.strip()
        .replace("```svg", "")
        .replace("```xml", "")
        .replace("```", "")
        .strip()
    )

    strict = re.search(r"<svg\b[\s\S]*?</svg>", text, flags=re.IGNORECASE)
    if strict:
        return strict.group(0).strip()

    start = text.lower().find("<svg")
    if start == -1:
        return None

    return text[start:].strip()

def repair_svg_candidate(text: str) -> str:
    candidate = extract_svg_candidate(text) or ""
    if not candidate:
        return ""

    candidate = trim_after_svg(candidate)
    candidate = trim_to_last_complete_tag(candidate).strip()

    if candidate.lower().count("<svg") == 0:
        return ""

    if "</svg>" not in candidate.lower():
        candidate = candidate.rstrip() + "</svg>"

    return candidate

def _round_num_token(s: str, decimals: int = 2) -> str:
    x = float(s)
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.{decimals}f}".rstrip("0").rstrip(".")

def round_svg_tree_numbers(root: ET.Element, decimals: int = 2) -> ET.Element:
    for el in root.iter():
        for k, v in list(el.attrib.items()):
            if k in _NUMERIC_ATTRS_SINGLE or k in _NUMERIC_ATTRS_MULTI:
                el.set(k, _num_re.sub(lambda m: _round_num_token(m.group(0), decimals), v))
    return root

def _style_text_is_unsafe(text: str) -> bool:
    t = (text or "").lower()
    if any(p in t for p in EXTERNAL_REF_PATTERNS):
        return True
    if "@import" in t or "expression(" in t or "javascript:" in t:
        return True
    return False

def _attr_is_unsafe(key: str, value: str) -> bool:
    key_lower = key.lower()
    value_lower = (value or "").lower().strip()

    if key_lower.startswith("on"):
        return True
    if key_lower in {"href", "xlink:href", "src"}:
        return not value_lower.startswith("#")
    if key_lower == "style" and _style_text_is_unsafe(value_lower):
        return True
    if any(p in value_lower for p in EXTERNAL_REF_PATTERNS):
        return True
    if "expression(" in value_lower:
        return True
    return False

def clean_text_content(tag: str, text: Optional[str]) -> str:
    if text is None:
        return ""
    if tag == "style":
        text = text[:CFG.MAX_STYLE_CHARS]
    else:
        text = text[:CFG.MAX_TEXT_CHARS]
    return text

def adapter_meta_path(adapter_dir: str) -> str:
    return os.path.join(adapter_dir, ADAPTER_META_FILENAME)

def load_adapter_meta(adapter_dir: str) -> Optional[dict]:
    path = adapter_meta_path(adapter_dir)
    if not os.path.exists(path):
        return None
    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None

def current_adapter_meta() -> dict:
    return {
        "platform": CFG.PLATFORM,
        "model_size": CFG.MODEL_SIZE,
        "base_model_path": CFG.BASE_MODEL_PATH,
        "target_modules": list(CFG.TARGET_MODULES),
        "lora_r": CFG.LORA_R,
        "lora_alpha": CFG.LORA_ALPHA,
        "lora_dropout": CFG.LORA_DROPOUT,
        "max_seq_len": CFG.MAX_SEQ_LEN,
        "tag_mode": CFG.TAG_MODE,
    }

def adapter_meta_matches_cfg(meta: Optional[dict]) -> bool:
    if not isinstance(meta, dict):
        return False
    return (
        meta.get("platform") == CFG.PLATFORM
        and meta.get("model_size") == CFG.MODEL_SIZE
        and meta.get("base_model_path") == CFG.BASE_MODEL_PATH
        and meta.get("target_modules") == list(CFG.TARGET_MODULES)
        and int(meta.get("lora_r", -1)) == int(CFG.LORA_R)
        and int(meta.get("lora_alpha", -1)) == int(CFG.LORA_ALPHA)
        and float(meta.get("lora_dropout", -999)) == float(CFG.LORA_DROPOUT)
        and int(meta.get("max_seq_len", -1)) == int(CFG.MAX_SEQ_LEN)
        and str(meta.get("tag_mode", "")).lower() == str(CFG.TAG_MODE).lower()
    )

def save_adapter_meta(adapter_dir: str) -> None:
    ensure_dir(adapter_dir)
    with open(adapter_meta_path(adapter_dir), "w", encoding="utf-8") as f:
        json.dump(current_adapter_meta(), f, indent=2, ensure_ascii=False)

def archive_dir(src_dir: str) -> Optional[str]:
    if not os.path.isdir(src_dir):
        return None
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = src_dir.rstrip("/\\") + f"_archived_{ts}"
    shutil.move(src_dir, dst)
    return dst

def cfg_snapshot() -> Dict[str, Any]:
    return {f.name: getattr(CFG, f.name) for f in dataclass_fields(CFG)}

@contextmanager
def temporary_cfg(**overrides):
    known = {f.name for f in dataclass_fields(CFG)}
    old = {}
    for k, v in overrides.items():
        if k not in known:
            raise KeyError(f"Unknown CFG field: {k}")
        old[k] = getattr(CFG, k)
        setattr(CFG, k, v)
    try:
        yield
    finally:
        for k, v in old.items():
            setattr(CFG, k, v)


def has_visible_shapes(svg: str, tag_mode: Optional[str] = None) -> bool:
    allowed = allowed_tags_for_mode(tag_mode)
    non_visual = {
        "svg", "g", "defs", "symbol", "clipPath", "mask",
        "linearGradient", "radialGradient", "stop",
        "title", "desc", "style", "pattern", "marker", "filter"
    }
    visual = allowed - non_visual
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return False

    for el in root.iter():
        tag = local_name(el.tag)
        if tag not in visual:
            continue
        if tag in {"text", "tspan"}:
            if (el.text or "").strip():
                return True
        else:
            return True
    return False

seed_everything(CFG.SEED)
ensure_dir(CFG.OUTPUT_DIR)

print("Current tag mode    :", CFG.TAG_MODE)
print("Allowed tags count  :", len(current_allowed_tags()))
print("Sanitizer mode      :", CFG.SANITIZER_MODE)
print("System prompt preview:")
print(build_system_prompt(CFG.TAG_MODE))


Current tag mode    : official
Allowed tags count  : 25
Sanitizer mode      : recover
System prompt preview:
You generate exactly one compact, valid SVG and nothing else. Output raw SVG only, not markdown, not code fences, not explanations. The output must start with <svg and end with </svg>. Canvas must be width='256' height='256' viewBox='0 0 256 256'. Allowed tags only: circle, clipPath, defs, desc, ellipse, filter, g, line, linearGradient, marker, mask, path, pattern, polygon, polyline, radialGradient, rect, stop, style, svg, symbol, text, title, tspan, use. You may use defs, gradients, text, tspan, clipPath, mask, use, pattern, marker, filter, title, and desc if they are necessary and remain valid. Prefer compact markup. Never use scripts, external URLs, animation, foreignObject, or event handlers.



## Why the sanitizer matters

Your sample rows already show several important patterns:
- many SVGs use `viewBox="0 0 200 200"`
- some use `24×24`, `100×100`, `128×128`, or `400×400`
- some rows come from other datasets and contain tags or styles you do **not** want under the current rules
- some rows contain empty `fill=""`, `currentColor`, or slightly messy attributes like `filling="0"`

A common mistake is to simply replace the root with `viewBox="0 0 256 256"` **without scaling the geometry**.
That makes `24×24` icons tiny in the upper-left corner and hurts the visual score.

The function below preserves geometry by wrapping the existing shapes in a scaled `<g>` when needed.


In [12]:

# Chunk 4.5 — helper note
# minify_svg / round_numeric_attributes / extract_svg_block 已在 Chunk 4 定义。
# 这一格保留为空，避免旧版本 notebook 里重复定义 helper 造成覆盖。
pass


In [13]:

# Chunk 5 — SVG sanitization and canvas normalization
import os
import re
import gc
import math
import json
import time
import shutil
import random
import datetime
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, Tuple, List, Dict, Any

try:
    from lxml import etree as LET
    HAS_LXML = True
except Exception:
    LET = None
    HAS_LXML = False

def parse_viewbox(vb: Optional[str]) -> Optional[Tuple[float, float, float, float]]:
    if not vb:
        return None
    parts = re.split(r"[ ,]+", vb.strip())
    if len(parts) != 4:
        return None
    try:
        x0, y0, w, h = map(float, parts)
        if w <= 0 or h <= 0:
            return None
        return x0, y0, w, h
    except Exception:
        return None

def maybe_wrap_scaled_group(clean_root: ET.Element, original_root: ET.Element) -> ET.Element:
    vb = parse_viewbox(original_root.attrib.get("viewBox"))
    if vb is None:
        return clean_root

    x0, y0, w, h = vb
    if abs(x0) < 1e-9 and abs(y0) < 1e-9 and abs(w - 256) < 1e-9 and abs(h - 256) < 1e-9:
        return clean_root

    sx = 256.0 / w
    sy = 256.0 / h

    children = list(clean_root)
    for child in children:
        clean_root.remove(child)

    g = ET.Element("g")
    if abs(x0) > 1e-9 or abs(y0) > 1e-9:
        g.set("transform", f"translate({-x0 * sx:.4f},{-y0 * sy:.4f}) scale({sx:.4f},{sy:.4f})")
    else:
        g.set("transform", f"scale({sx:.4f},{sy:.4f})")

    for child in children:
        g.append(child)
    clean_root.append(g)
    return clean_root

def _parse_svg_candidate(svg_text: str, sanitizer_mode: Optional[str] = None):
    mode = (sanitizer_mode or CFG.SANITIZER_MODE).lower().strip()

    try:
        return ET.fromstring(svg_text), "xml_ok"
    except ET.ParseError:
        if mode != "recover":
            return None, "xml_parse_error"

    repaired = repair_svg_candidate(svg_text)
    if repaired and repaired != svg_text:
        try:
            return ET.fromstring(repaired), "xml_recovered_by_trim"
        except ET.ParseError:
            pass

    if mode == "recover" and HAS_LXML:
        try:
            parser = LET.XMLParser(recover=True, remove_comments=True)
            root = LET.fromstring(repaired.encode("utf-8") if repaired else svg_text.encode("utf-8"), parser=parser)
            xml_text = LET.tostring(root, encoding="unicode")
            return ET.fromstring(xml_text), "xml_recovered_by_lxml"
        except Exception:
            pass

    return None, "xml_parse_error"

def sanitize_svg_with_reason(
    raw_text: str,
    max_chars: int = CFG.MAX_SVG_CHARS,
    sanitizer_mode: Optional[str] = None,
    tag_mode: Optional[str] = None,
) -> Tuple[Optional[str], str]:
    mode = (sanitizer_mode or CFG.SANITIZER_MODE).lower().strip()
    tag_mode = (tag_mode or CFG.TAG_MODE).lower().strip()

    candidate = extract_svg_candidate(raw_text)
    if candidate is None:
        return None, "no_svg_block"

    candidate = trim_after_svg(candidate).strip()
    if mode == "recover":
        candidate = repair_svg_candidate(candidate)

    root, parse_reason = _parse_svg_candidate(candidate, sanitizer_mode=mode)
    if root is None:
        return None, parse_reason

    if local_name(root.tag) != "svg":
        return None, "root_not_svg"

    allowed_tags = allowed_tags_for_mode(tag_mode)

    def clean_node(node: ET.Element) -> Optional[ET.Element]:
        tag = local_name(node.tag)
        if tag in ALWAYS_BLOCKED_TAGS or tag not in allowed_tags:
            return None

        new_node = ET.Element(tag)

        for k, v in node.attrib.items():
            if not _attr_is_unsafe(k, v):
                new_node.set(k, v)

        if tag in TEXT_LIKE_TAGS and node.text:
            txt = clean_text_content(tag, node.text)
            if tag == "style" and _style_text_is_unsafe(txt):
                txt = ""
            new_node.text = txt

        for child in list(node):
            cleaned = clean_node(child)
            if cleaned is not None:
                new_node.append(cleaned)

        return new_node

    clean_root = clean_node(root)
    if clean_root is None:
        return None, "clean_root_none"

    clean_root = maybe_wrap_scaled_group(clean_root, root)
    clean_root.tag = "svg"
    clean_root.set("xmlns", "http://www.w3.org/2000/svg")
    clean_root.set("width", "256")
    clean_root.set("height", "256")
    clean_root.set("viewBox", "0 0 256 256")

    clean_root = round_svg_tree_numbers(clean_root, decimals=2)
    svg_text = ET.tostring(clean_root, encoding="unicode", method="xml")
    svg_text = minify_svg(svg_text)

    if len(svg_text) > max_chars:
        return None, "too_long"

    try:
        root2 = ET.fromstring(svg_text)
    except ET.ParseError as e:
        return None, f"final_parse_error:{e}"

    path_count = sum(1 for el in root2.iter() if local_name(el.tag) == "path")
    if path_count > CFG.MAX_PATHS:
        return None, "too_many_paths"

    if local_name(root2.tag) != "svg":
        return None, "bad_prefix"

    for el in root2.iter():
        tag = local_name(el.tag)
        if tag not in allowed_tags:
            return None, f"tag_not_allowed:{tag}"
        if tag in ALWAYS_BLOCKED_TAGS:
            return None, f"tag_blocked:{tag}"
        if tag == "style" and _style_text_is_unsafe(el.text or ""):
            return None, "unsafe_style_text"
        for k, v in el.attrib.items():
            if _attr_is_unsafe(k, v):
                return None, f"unsafe_attr:{k}"

    return svg_text, ("ok" if parse_reason == "xml_ok" else parse_reason)

def sanitize_svg(
    raw_text: str,
    max_chars: int = CFG.MAX_SVG_CHARS,
    sanitizer_mode: Optional[str] = None,
    tag_mode: Optional[str] = None,
) -> Optional[str]:
    svg, _ = sanitize_svg_with_reason(
        raw_text,
        max_chars=max_chars,
        sanitizer_mode=sanitizer_mode,
        tag_mode=tag_mode,
    )
    return svg

def is_valid_svg(svg: str, tag_mode: Optional[str] = None) -> bool:
    allowed_tags = allowed_tags_for_mode(tag_mode)
    if not isinstance(svg, str) or len(svg) > CFG.MAX_SVG_CHARS:
        return False
    try:
        root = ET.fromstring(svg)
    except ET.ParseError:
        return False
    if local_name(root.tag) != "svg":
        return False
    path_count = 0
    for el in root.iter():
        tag = local_name(el.tag)
        if tag not in allowed_tags or tag in ALWAYS_BLOCKED_TAGS:
            return False
        if tag == "path":
            path_count += 1
        if tag == "style" and _style_text_is_unsafe(el.text or ""):
            return False
        for k, v in el.attrib.items():
            if _attr_is_unsafe(k, v):
                return False
    return path_count <= CFG.MAX_PATHS

print("HAS_LXML recover parser:", HAS_LXML)


HAS_LXML recover parser: True


In [14]:

# Chunk 6 — smoke tests for the sanitizer
samples = [
    '<svg width="24" height="24" viewBox="0 0 24 24" xmlns="http://www.w3.org/2000/svg"><rect x="2" y="2" width="20" height="20"/></svg>',
    '<svg width="200" height="200" viewBox="0 0 200 200" xmlns="http://www.w3.org/2000/svg"><circle cx="100" cy="100" r="40"/></svg>',
]

for i, s in enumerate(samples, 1):
    clean_svg, reason = sanitize_svg_with_reason(s)
    print(f"sample {i} -> reason = {reason}")
    print(clean_svg)
    print("valid:", is_valid_svg(clean_svg))
    print("-" * 100)


sample 1 -> reason = ok
<svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg"><g transform="scale(10.67,10.67)"><rect x="2" y="2" width="20" height="20" /></g></svg>
valid: True
----------------------------------------------------------------------------------------------------
sample 2 -> reason = ok
<svg width="256" height="256" viewBox="0 0 256 256" xmlns="http://www.w3.org/2000/svg"><g transform="scale(1.28,1.28)"><circle cx="100" cy="100" r="40" /></g></svg>
valid: True
----------------------------------------------------------------------------------------------------



## Load raw data and inspect a few rows

This is useful because the training set is not perfectly homogeneous.
You already showed examples with:
- pure icon rows
- large illustration rows
- imported rows like `thesantatitan_deepseek-svg-dataset_*`
- highly detailed rows such as people / emojis / complex objects

That heterogeneity matters later when you decide whether to filter noisy rows.


In [15]:

# Chunk 7 — load raw CSVs and inspect
raw_train_df = pd.read_csv(CFG.TRAIN_CSV)
raw_test_df = pd.read_csv(CFG.TEST_CSV)

print("raw train shape:", raw_train_df.shape)
print("raw test shape:", raw_test_df.shape)
print("train columns:", raw_train_df.columns.tolist())
print("test columns:", raw_test_df.columns.tolist())

display(raw_train_df.head(5))


raw train shape: (50000, 3)
raw test shape: (1000, 2)
train columns: ['id', 'prompt', 'svg']
test columns: ['id', 'prompt']


,id,prompt,svg
0,fd61e324e0cec5c11f337d0bfe2858c8,The image features two orange squares with a m...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
1,999b3d4d5a860725bf9528910b5612f3,A simple smiley face with a wide open mouth an...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
2,1aaa84517819c25f783ae1c0cb337fc5,The image features a black-outlined icon of a ...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
3,919a7da8bd44dc7781dbe87383a268cc,The image displays a black icon with a photo-l...,"<svg xmlns=""http://www.w3.org/2000/svg"" viewBo..."
4,thesantatitan_deepseek-svg-dataset_0000581,Generate svg code for an image that looks like...,"<svg width=""24"" height=""24"" viewBox=""0 0 24 24..."


In [16]:
# Chunk 8 — clean the train set and collect cleaning stats

def maybe_limit(df: pd.DataFrame, n: Optional[int], seed: int) -> pd.DataFrame:
    if n is None or len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

def load_competition_data() -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    train_df = pd.read_csv(CFG.TRAIN_CSV)
    test_df = pd.read_csv(CFG.TEST_CSV)

    expected_train_cols = ["id", "prompt", "svg"]
    expected_test_cols = ["id", "prompt"]
    if train_df.columns.tolist() != expected_train_cols:
        raise ValueError(f"Unexpected train columns: {train_df.columns.tolist()}")
    if test_df.columns.tolist() != expected_test_cols:
        raise ValueError(f"Unexpected test columns: {test_df.columns.tolist()}")

    train_df["prompt"] = train_df["prompt"].fillna("").astype(str).str.strip()
    train_df["svg"] = train_df["svg"].fillna("").astype(str)
    test_df["prompt"] = test_df["prompt"].fillna("").astype(str).str.strip()

    train_df["svg_raw"] = train_df["svg"].astype(str)

    clean_results = train_df["svg_raw"].map(
        lambda x: sanitize_svg_with_reason(
            x,
            max_chars=CFG.MAX_SVG_CHARS,
            sanitizer_mode=CFG.SANITIZER_MODE,
            tag_mode=CFG.TAG_MODE,
        )
    )
    train_df["svg_clean"] = clean_results.map(lambda x: x[0])
    train_df["clean_reason"] = clean_results.map(lambda x: x[1])

    train_df = train_df[train_df["svg_clean"].notna()].copy().reset_index(drop=True)
    train_df["svg"] = train_df["svg_clean"].astype(str)
    train_df["clean_changed"] = train_df["svg_raw"].astype(str) != train_df["svg"].astype(str)
    train_df["visible_ok"] = train_df["svg"].map(
        lambda x: bool(is_valid_svg(x, tag_mode=CFG.TAG_MODE) and has_visible_shapes(x, tag_mode=CFG.TAG_MODE))
    )

    clean_report = (
        train_df.groupby(["clean_reason", "visible_ok", "clean_changed"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    return train_df, test_df, clean_report

train_df, test_df, clean_report = load_competition_data()

print("clean train rows:", len(train_df))
print("test rows:", len(test_df))
print("visible share   :", round(train_df["visible_ok"].mean(), 4))
display(clean_report.head(20))

clean train rows: 49999
test rows: 1000
visible share   : 1.0


,clean_reason,visible_ok,clean_changed,count
0,ok,True,True,49998
1,ok,False,True,1


In [17]:
# Chunk 8.5 — complexity-aware filtering for retrieval / training
from collections import Counter

_PATH_D_RE = re.compile(r"<path\b[^>]*\sd=(['\"])(.*?)\1", flags=re.IGNORECASE | re.DOTALL)
_PATH_CMD_RE = re.compile(r"[MmLlHhVvCcSsQqTtAaZz]")
_SHAPE_TAGS = ("path", "rect", "circle", "ellipse", "line", "polyline", "polygon")

def extract_path_ds(svg: str) -> List[str]:
    if not isinstance(svg, str) or not svg:
        return []
    return [m[1] for m in _PATH_D_RE.findall(svg)]

def summarize_path_complexity(svg: str) -> Dict[str, Any]:
    path_ds = extract_path_ds(svg)
    cmd_counter = Counter()
    path_d_lens = []
    control_point_proxy = 0

    for d in path_ds:
        cmds = _PATH_CMD_RE.findall(d)
        path_d_lens.append(len(d))
        upper_cmds = [c.upper() for c in cmds]
        cmd_counter.update(upper_cmds)
        control_point_proxy += (
            3 * sum(c == "C" for c in upper_cmds)
            + 2 * sum(c == "S" for c in upper_cmds)
            + 2 * sum(c == "Q" for c in upper_cmds)
            + 1 * sum(c == "T" for c in upper_cmds)
            + 2 * sum(c == "A" for c in upper_cmds)
        )

    return {
        "command_count": int(sum(cmd_counter.values())),
        "command_diversity": int(len(cmd_counter)),
        "control_point_proxy": int(control_point_proxy),
        "mean_path_d_len": float(np.mean(path_d_lens)) if len(path_d_lens) > 0 else 0.0,
        "max_path_d_len": int(max(path_d_lens)) if len(path_d_lens) > 0 else 0,
    }

def add_svg_stats(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # 先删掉旧的派生列，避免重复运行时出现同名双列
    derived_cols = [
        "svg_len", "path_count", "prompt_word_len", "prompt_char_len",
        "is_external_like",
        "command_count", "command_diversity", "control_point_proxy",
        "mean_path_d_len", "max_path_d_len",
        "primitive_type_count", "has_transform", "has_defs", "has_style",
        "has_text", "has_use", "has_clip_or_mask", "has_gradient_or_filter",
        "has_currentColor",
    ]
    drop_cols = [c for c in derived_cols if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df["svg_len"] = df["svg"].astype(str).str.len()
    df["path_count"] = df["svg"].astype(str).str.count(r"<path\b")
    df["prompt_word_len"] = df["prompt"].astype(str).str.split().str.len()
    df["prompt_char_len"] = df["prompt"].astype(str).str.len()

    df["is_external_like"] = df["id"].astype(str).str.contains(
        r"deepseek|thesantatitan|emoji|openmoji", case=False, regex=True
    )

    path_stats = df["svg"].map(summarize_path_complexity).apply(pd.Series)
    for col in ["command_count", "command_diversity", "control_point_proxy", "mean_path_d_len", "max_path_d_len"]:
        df[col] = path_stats[col]

    df["primitive_type_count"] = df["svg"].astype(str).map(
        lambda x: sum(bool(re.search(fr"<{tag}\b", x, flags=re.IGNORECASE)) for tag in _SHAPE_TAGS)
    )
    df["has_transform"] = df["svg"].astype(str).str.contains(r"\btransform\s*=", case=False, regex=True)
    df["has_defs"] = df["svg"].astype(str).str.contains(r"<defs\b", case=False, regex=True)
    df["has_style"] = df["svg"].astype(str).str.contains(r"<style\b|\bstyle\s*=", case=False, regex=True)
    df["has_text"] = df["svg"].astype(str).str.contains(r"<text\b|<tspan\b", case=False, regex=True)
    df["has_use"] = df["svg"].astype(str).str.contains(r"<use\b", case=False, regex=True)
    df["has_clip_or_mask"] = df["svg"].astype(str).str.contains(r"<clipPath\b|<mask\b", case=False, regex=True)
    df["has_gradient_or_filter"] = df["svg"].astype(str).str.contains(
        r"<linearGradient\b|<radialGradient\b|<filter\b", case=False, regex=True
    )
    df["has_currentColor"] = df["svg"].astype(str).str.contains("currentColor", regex=False)

    if "visible_ok" not in df.columns:
        df["visible_ok"] = df["svg"].map(
            lambda x: bool(is_valid_svg(x, tag_mode=CFG.TAG_MODE) and has_visible_shapes(x, tag_mode=CFG.TAG_MODE))
        )

    return df

train_df = add_svg_stats(train_df)

summary_cols = [
    "svg_len", "path_count", "prompt_word_len",
    "command_count", "command_diversity", "control_point_proxy",
    "mean_path_d_len", "max_path_d_len", "primitive_type_count", "visible_ok",
]
display(train_df[summary_cols].describe(include="all"))

filter_checks = pd.DataFrame({
    "svg_len_ok": train_df["svg_len"] <= CFG.FILTER_MAX_SVG_LEN,
    "path_count_ok": train_df["path_count"] <= CFG.FILTER_MAX_PATHS,
    "prompt_word_len_ok": train_df["prompt_word_len"].between(CFG.FILTER_MIN_PROMPT_WORDS, CFG.FILTER_MAX_PROMPT_WORDS),
    "command_count_ok": train_df["command_count"] <= CFG.FILTER_MAX_COMMANDS,
    "control_points_ok": train_df["control_point_proxy"] <= CFG.FILTER_MAX_CONTROL_POINTS,
    "command_diversity_ok": train_df["command_diversity"] <= CFG.FILTER_MAX_COMMAND_DIVERSITY,
    "primitive_types_ok": train_df["primitive_type_count"] <= CFG.FILTER_MAX_PRIMITIVE_TYPES,
    "mean_path_d_len_ok": train_df["mean_path_d_len"] <= CFG.FILTER_MAX_MEAN_PATH_D_LEN,
    "max_path_d_len_ok": train_df["max_path_d_len"] <= CFG.FILTER_MAX_MAX_PATH_D_LEN,
    "not_external_like": ~train_df["is_external_like"],
    "visible_ok": train_df["visible_ok"] if CFG.FILTER_REQUIRE_VISIBLE else True,
    "no_text": (~train_df["has_text"]) if not CFG.FILTER_ALLOW_TEXT else True,
    "no_use": (~train_df["has_use"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
    "no_clip_mask": (~train_df["has_clip_or_mask"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
    "no_grad_filter": (~train_df["has_gradient_or_filter"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
})

filter_mask = filter_checks.all(axis=1)
filtered_train_df = train_df[filter_mask].copy().reset_index(drop=True)

filter_pass_rate = filter_checks.mean().sort_values(ascending=True).rename("pass_rate").reset_index()
filter_pass_rate.columns = ["rule", "pass_rate"]

print("original train:", len(train_df))
print("filtered train:", len(filtered_train_df))
print("filtered share:", round(len(filtered_train_df) / max(len(train_df), 1), 4))
display(filter_pass_rate)
display(filtered_train_df[[
    "id", "prompt", "svg_len", "path_count", "prompt_word_len",
    "command_count", "command_diversity", "control_point_proxy",
    "primitive_type_count", "max_path_d_len"
]].head(5))

,svg_len,path_count,prompt_word_len,command_count,command_diversity,control_point_proxy,mean_path_d_len,max_path_d_len,primitive_type_count,visible_ok
count,49999.000000,49999.000000,49999.000000,49999.000000,49999.000000,49999.000000,49999.000000,49999.000000,49999.000000,49999
unique,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
top,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,True
freq,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,49998
mean,1282.200064,2.414088,19.719574,46.904578,4.300046,53.483650,605.928181,724.490370,1.085662,NaN
std,873.787352,3.152709,10.334305,34.751539,1.177128,48.509471,553.128077,593.372366,0.351350,NaN
min,150.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN
25%,708.000000,1.000000,12.000000,25.000000,4.000000,20.000000,240.000000,327.500000,1.000000,NaN
50%,1088.000000,1.000000,17.000000,41.000000,4.000000,42.000000,454.000000,594.000000,1.000000,NaN
75%,1650.000000,3.000000,24.000000,62.000000,5.000000,76.000000,808.166667,973.000000,1.000000,NaN


original train: 49999
filtered train: 32954
filtered share: 0.6591


,rule,pass_rate
0,mean_path_d_len_ok,0.790236
1,max_path_d_len_ok,0.921578
2,prompt_word_len_ok,0.928959
3,not_external_like,0.945619
4,control_points_ok,0.971159
5,no_grad_filter,0.990100
6,no_clip_mask,0.992280
7,svg_len_ok,0.992820
8,command_count_ok,0.994340
9,no_text,0.995740


,id,prompt,svg_len,path_count,prompt_word_len,command_count,command_diversity,control_point_proxy,primitive_type_count,max_path_d_len
0,fd61e324e0cec5c11f337d0bfe2858c8,The image features two orange squares with a m...,1589,3,20,101.0,4.0,33.0,1,779.0
1,999b3d4d5a860725bf9528910b5612f3,A simple smiley face with a wide open mouth an...,1354,4,12,42.0,4.0,60.0,1,342.0
2,919a7da8bd44dc7781dbe87383a268cc,The image displays a black icon with a photo-l...,1631,2,32,60.0,4.0,81.0,1,733.0
3,d342c3ca-4c5a-492f-9a93-1604bcb9a4b2,An arrow pointing right emerges from a line lo...,643,2,13,17.0,5.0,26.0,1,204.0
4,977cd8ad18ae20da42de59594aafeaf3,The image features a teal circular outline con...,2227,4,20,79.0,5.0,112.0,1,554.0


In [18]:
# Chunk 8.6 — canonicalization + easy/medium/hard curriculum buckets
# 目标：
# 1) 训练前把“同一种图”的 markup 尽量规范化，减少无意义的结构噪声
# 2) 生成 easy / medium / hard / hard_focus 四类训练源，支持 curriculum A/B

import re
import math
import copy
import xml.etree.ElementTree as ET

_NUM_TOKEN_RE = re.compile(r"(?<![A-Za-z])[-+]?(?:\d+\.\d+|\d+|\.\d+)(?:[eE][-+]?\d+)?")
_HEX6_RE = re.compile(r"^#([0-9a-fA-F]{6})$")
_RGB_RE = re.compile(r"rgb\(\s*([0-9]{1,3})\s*,\s*([0-9]{1,3})\s*,\s*([0-9]{1,3})\s*\)$", re.I)
_SPACES_RE = re.compile(r"\s+")
_PATH_LIKE_ATTRS = {"d", "points", "viewBox", "x", "y", "x1", "x2", "y1", "y2", "cx", "cy", "r", "rx", "ry", "width", "height", "stroke-width", "opacity", "fill-opacity", "stroke-opacity", "font-size"}

PREFERRED_ATTR_ORDER = [
    "xmlns", "width", "height", "viewBox", "fill", "stroke", "stroke-width",
    "opacity", "fill-opacity", "stroke-opacity", "d", "points", "x", "y",
    "x1", "y1", "x2", "y2", "cx", "cy", "r", "rx", "ry", "transform", "id",
    "class", "style"
]

def _fmt_num_token(token: str, decimals: int = 2) -> str:
    try:
        x = float(token)
    except Exception:
        return token
    if abs(x) < 1e-12:
        x = 0.0
    x = round(x, decimals)
    if abs(x - int(round(x))) < 1e-12:
        return str(int(round(x)))
    out = f"{x:.{decimals}f}".rstrip("0").rstrip(".")
    return out if out != "-0" else "0"

def _normalize_numeric_string(s: str, decimals: int = 2) -> str:
    if not isinstance(s, str) or s == "":
        return s
    return _NUM_TOKEN_RE.sub(lambda m: _fmt_num_token(m.group(0), decimals=decimals), s)

def _compress_hex_color(v: str) -> str:
    m = _HEX6_RE.match(v.strip())
    if not m:
        return v
    h = m.group(1).lower()
    if h[0] == h[1] and h[2] == h[3] and h[4] == h[5]:
        return f"#{h[0]}{h[2]}{h[4]}"
    return f"#{h}"

def _normalize_color(v: str) -> str:
    if not isinstance(v, str):
        return v
    s = v.strip()
    if s == "":
        return s
    low = s.lower()
    if low in {"none", "transparent", "currentcolor"}:
        return low
    m = _RGB_RE.match(low)
    if m:
        rgb = [max(0, min(255, int(g))) for g in m.groups()]
        return _compress_hex_color("#{0:02x}{1:02x}{2:02x}".format(*rgb))
    if low.startswith("#"):
        return _compress_hex_color(low)
    return low

def _normalize_style_attr(style_value: str, decimals: int = 2) -> str:
    if not isinstance(style_value, str) or style_value.strip() == "":
        return style_value
    pairs = []
    for part in style_value.split(";"):
        if ":" not in part:
            continue
        k, v = part.split(":", 1)
        k = k.strip().lower()
        v = v.strip()
        if k in {"fill", "stroke", "stop-color", "color"}:
            v = _normalize_color(v)
        else:
            v = _normalize_numeric_string(v, decimals=decimals)
        if k in {"opacity", "fill-opacity", "stroke-opacity"} and v == "1":
            continue
        pairs.append((k, v))
    pairs = sorted(dict(pairs).items(), key=lambda kv: kv[0])
    return ";".join(f"{k}:{v}" for k, v in pairs)

def _maybe_strip_redundant_attr(tag: str, key: str, val: str) -> bool:
    if key in {"xmlns:svg", "xml:space"}:
        return True
    if key in {"opacity", "fill-opacity", "stroke-opacity"} and str(val).strip() == "1":
        return True
    if key == "stroke-width" and str(val).strip() in {"0", "0.0"}:
        return True
    if key == "version" and str(val).strip() == "1.1":
        return True
    return False

def _attr_sort_key(key: str):
    if key in PREFERRED_ATTR_ORDER:
        return (0, PREFERRED_ATTR_ORDER.index(key))
    return (1, key)

def canonicalize_svg_markup(svg_text: str) -> Optional[str]:
    if not isinstance(svg_text, str) or not svg_text.strip():
        return None
    try:
        root = ET.fromstring(svg_text)
    except Exception:
        return None

    def _walk(el: ET.Element, is_root: bool = False) -> None:
        # normalize child whitespace
        if el.text is not None and not el.text.strip():
            el.text = None
        if el.tail is not None and not el.tail.strip():
            el.tail = None

        raw_attrs = dict(el.attrib)
        norm_items = []

        for k, v in raw_attrs.items():
            lk = k.split("}", 1)[-1] if "}" in k else k
            vv = str(v).strip()

            if lk == "style" and getattr(CFG, "CANON_NORMALIZE_STYLE", False):
                vv = _normalize_style_attr(vv, decimals=CFG.CANON_DECIMALS)
            elif lk in {"fill", "stroke", "stop-color", "color"} and getattr(CFG, "CANON_NORMALIZE_COLORS", True):
                vv = _normalize_color(vv)
            elif lk in _PATH_LIKE_ATTRS or any(ch.isdigit() for ch in vv):
                vv = _normalize_numeric_string(vv, decimals=CFG.CANON_DECIMALS)

            if getattr(CFG, "CANON_REMOVE_REDUNDANT_ATTRS", True) and _maybe_strip_redundant_attr(el.tag, lk, vv):
                continue

            norm_items.append((lk, vv))

        if is_root:
            # competition canvas is fixed 256x256
            forced = {
                "xmlns": "http://www.w3.org/2000/svg",
                "width": "256",
                "height": "256",
                "viewBox": "0 0 256 256",
            }
            # remove duplicates that conflict with root normalization
            norm_items = [(k, v) for (k, v) in norm_items if k not in forced]
            norm_items = list(forced.items()) + norm_items

        norm_items = sorted(norm_items, key=lambda kv: _attr_sort_key(kv[0]))
        el.attrib.clear()
        for k, v in norm_items:
            el.set(k, v)

        for child in list(el):
            _walk(child, is_root=False)

        if getattr(CFG, "CANON_COLLAPSE_GROUPS", False):
            children = list(el)
            if (
                len(children) == 1
                and len(el.attrib) == 0
                and (el.text is None)
                and (el.tag.split("}", 1)[-1] if "}" in el.tag else el.tag) == "g"
            ):
                only = children[0]
                el.clear()
                el.tag = only.tag
                for k, v in only.attrib.items():
                    el.set(k, v)
                el.text = only.text
                el.tail = only.tail
                for gc in list(only):
                    el.append(gc)

    _walk(root, is_root=True)

    svg_out = ET.tostring(root, encoding="unicode", method="xml")
    svg_out = re.sub(r">\s+<", "><", svg_out)
    svg_out = _SPACES_RE.sub(" ", svg_out).strip()
    svg_out = trim_after_svg(svg_out)
    return svg_out

def apply_optional_canonicalization(df: pd.DataFrame) -> pd.DataFrame:
    work = df.copy()
    work["svg_before_canon"] = work["svg"].astype(str)
    if getattr(CFG, "ENABLE_CANONICALIZATION", False):
        canon_svg = work["svg"].map(canonicalize_svg_markup)
        canon_svg = canon_svg.fillna(work["svg"])
        # canonicalization之后再走一次sanitize，避免格式调整后引入边角问题
        canon_svg = canon_svg.map(
            lambda x: sanitize_svg(
                x,
                max_chars=CFG.MAX_SVG_CHARS,
                sanitizer_mode=CFG.SANITIZER_MODE,
                tag_mode=CFG.TAG_MODE,
            )
        )
        canon_svg = canon_svg.fillna(work["svg"])
        work["svg"] = canon_svg.astype(str)
    work["canon_changed"] = work["svg_before_canon"].astype(str) != work["svg"].astype(str)
    return work

train_df = apply_optional_canonicalization(train_df)
train_df = add_svg_stats(train_df)

def classify_prompt_family(prompt: str) -> str:
    p = str(prompt).lower()
    if any(w in p for w in ["inside", "next to", "on top of", "under", "over", "holding", "between", "with ", "behind", "in front of"]):
        return "composite_relation"
    if any(w in p for w in ["person", "man", "woman", "boy", "girl", "face", "hand", "tree", "flower", "leaf", "animal", "cat", "dog", "bird", "fish"]):
        return "person_organic"
    if any(w in p for w in ["phone", "laptop", "computer", "camera", "monitor", "watch", "robot", "car", "truck", "bike", "train", "plane"]):
        return "device_object"
    if any(w in p for w in ["frame", "window", "door", "box", "container", "screen", "card", "poster", "badge"]):
        return "container_frame"
    return "simple_object"

train_df["prompt_family"] = train_df["prompt"].map(classify_prompt_family)

STRICT_PROFILE = {
    "svg_len": 4200,
    "path_count": 72,
    "prompt_min": 4,
    "prompt_max": 36,
    "command_count": 180,
    "control_points": 160,
    "command_diversity": 8,
    "primitive_types": 4,
    "mean_path_d_len": 900,
    "max_path_d_len": 1600,
}

MEDIUM_PROFILE = {
    "svg_len": 7000,
    "path_count": 120,
    "prompt_min": 4,
    "prompt_max": 40,
    "command_count": 300,
    "control_points": 260,
    "command_diversity": 8,
    "primitive_types": 5,
    "mean_path_d_len": 1400,
    "max_path_d_len": 2800,
}

HARD_PROFILE = {
    "svg_len": 11000,
    "path_count": 180,
    "prompt_min": 3,
    "prompt_max": 48,
    "command_count": 520,
    "control_points": 420,
    "command_diversity": 9,
    "primitive_types": 6,
    "mean_path_d_len": 2200,
    "max_path_d_len": 4500,
}

def build_filter_checks(df: pd.DataFrame, profile: Dict[str, Any]) -> pd.DataFrame:
    return pd.DataFrame({
        "svg_len_ok": df["svg_len"] <= profile["svg_len"],
        "path_count_ok": df["path_count"] <= profile["path_count"],
        "prompt_word_len_ok": df["prompt_word_len"].between(profile["prompt_min"], profile["prompt_max"]),
        "command_count_ok": df["command_count"] <= profile["command_count"],
        "control_points_ok": df["control_point_proxy"] <= profile["control_points"],
        "command_diversity_ok": df["command_diversity"] <= profile["command_diversity"],
        "primitive_types_ok": df["primitive_type_count"] <= profile["primitive_types"],
        "mean_path_d_len_ok": df["mean_path_d_len"] <= profile["mean_path_d_len"],
        "max_path_d_len_ok": df["max_path_d_len"] <= profile["max_path_d_len"],
        "not_external_like": ~df["is_external_like"],
        "visible_ok": df["visible_ok"] if CFG.FILTER_REQUIRE_VISIBLE else True,
        "no_text": (~df["has_text"]) if not CFG.FILTER_ALLOW_TEXT else True,
        "no_use": (~df["has_use"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
        "no_clip_mask": (~df["has_clip_or_mask"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
        "no_grad_filter": (~df["has_gradient_or_filter"]) if not CFG.FILTER_ALLOW_COMPLEX_EFFECTS else True,
    })

easy_checks = build_filter_checks(train_df, STRICT_PROFILE)
medium_checks = build_filter_checks(train_df, MEDIUM_PROFILE)
hard_checks = build_filter_checks(train_df, HARD_PROFILE)

filtered_easy_df = train_df[easy_checks.all(axis=1)].copy().reset_index(drop=True)
filtered_medium_df = train_df[medium_checks.all(axis=1)].copy().reset_index(drop=True)
filtered_hard_df = train_df[hard_checks.all(axis=1)].copy().reset_index(drop=True)

# 保持 backward compatible：filtered_train_df 仍视为 strict/easy
filtered_train_df = filtered_easy_df.copy().reset_index(drop=True)

# hard focus: 复杂图 + 历史上容易掉分的 family
cmd_q = float(train_df["command_count"].quantile(0.75))
ctrl_q = float(train_df["control_point_proxy"].quantile(0.75))
path_q = float(train_df["path_count"].quantile(0.75))

hard_focus_mask = (
    (filtered_hard_df["prompt_family"].isin(getattr(CFG, "CURRICULUM_STAGEC_FAMILIES", ())))
    | (filtered_hard_df["command_count"] >= cmd_q)
    | (filtered_hard_df["control_point_proxy"] >= ctrl_q)
    | (filtered_hard_df["path_count"] >= path_q)
)

hard_focus_df = filtered_hard_df[hard_focus_mask].copy().reset_index(drop=True)

curriculum_stage_a_df = (
    pd.concat([filtered_easy_df, filtered_medium_df], axis=0)
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)

curriculum_stage_b_df = (
    pd.concat([filtered_easy_df, filtered_medium_df, filtered_hard_df], axis=0)
    .drop_duplicates(subset=["id"], keep="first")
    .reset_index(drop=True)
)

bucket_summary = pd.DataFrame([
    {"source": "train_full", "rows": len(train_df), "svg_len_mean": float(train_df["svg_len"].mean()), "path_mean": float(train_df["path_count"].mean())},
    {"source": "filtered_easy", "rows": len(filtered_easy_df), "svg_len_mean": float(filtered_easy_df["svg_len"].mean()), "path_mean": float(filtered_easy_df["path_count"].mean())},
    {"source": "filtered_medium", "rows": len(filtered_medium_df), "svg_len_mean": float(filtered_medium_df["svg_len"].mean()), "path_mean": float(filtered_medium_df["path_count"].mean())},
    {"source": "filtered_hard", "rows": len(filtered_hard_df), "svg_len_mean": float(filtered_hard_df["svg_len"].mean()), "path_mean": float(filtered_hard_df["path_count"].mean())},
    {"source": "hard_focus", "rows": len(hard_focus_df), "svg_len_mean": float(hard_focus_df["svg_len"].mean()), "path_mean": float(hard_focus_df["path_count"].mean())},
    {"source": "curriculum_stage_a", "rows": len(curriculum_stage_a_df), "svg_len_mean": float(curriculum_stage_a_df["svg_len"].mean()), "path_mean": float(curriculum_stage_a_df["path_count"].mean())},
    {"source": "curriculum_stage_b", "rows": len(curriculum_stage_b_df), "svg_len_mean": float(curriculum_stage_b_df["svg_len"].mean()), "path_mean": float(curriculum_stage_b_df["path_count"].mean())},
]).sort_values("rows", ascending=False).reset_index(drop=True)

print("canonicalization enabled:", getattr(CFG, "ENABLE_CANONICALIZATION", False))
print("canon changed share      :", round(train_df["canon_changed"].mean(), 4))
display(bucket_summary)
display(train_df["prompt_family"].value_counts(dropna=False).rename_axis("prompt_family").reset_index(name="count"))

canonicalization enabled: False
canon changed share      : 0.0


,source,rows,svg_len_mean,path_mean
0,train_full,49999,1282.200064,2.414088
1,curriculum_stage_b,44513,1250.117449,2.414306
2,filtered_hard,44513,1250.117449,2.414306
3,curriculum_stage_a,40095,1187.851104,2.489039
4,filtered_medium,40095,1187.851104,2.489039
5,hard_focus,36688,1362.493840,2.673899
6,filtered_easy,32954,1092.493324,2.612642


,prompt_family,count
0,composite_relation,34464
1,simple_object,12511
2,person_organic,1482
3,container_frame,855
4,device_object,687


In [19]:
# Chunk 9 — quick distribution checks after cleaning / filtering

if "prompt_char_len" not in train_df.columns or "prompt_word_len" not in train_df.columns:
    train_df["prompt_char_len"] = train_df["prompt"].astype(str).str.len()
    train_df["prompt_word_len"] = train_df["prompt"].astype(str).str.split().str.len()

train_df["source_group"] = np.where(train_df["id"].str.contains("deepseek-svg-dataset", regex=False), "external_like", "main_like")

print(train_df[[
    "prompt_char_len", "prompt_word_len", "svg_len", "path_count",
    "command_count", "control_point_proxy", "primitive_type_count"
]].describe())
print("share with currentColor:", round(train_df["has_currentColor"].mean(), 4))
print("share visible_ok      :", round(train_df["visible_ok"].mean(), 4))
print(train_df["source_group"].value_counts())

display(train_df[[
    "id", "prompt", "svg_len", "path_count", "command_count",
    "control_point_proxy", "primitive_type_count", "source_group"
]].sample(5, random_state=CFG.SEED))

       prompt_char_len  prompt_word_len       svg_len    path_count  \
count     49999.000000     49999.000000  49999.000000  49999.000000   
mean        116.565251        19.719574   1282.200064      2.414088   
std          64.188064        10.334305    873.787352      3.152709   
min           5.000000         1.000000    150.000000      0.000000   
25%          72.000000        12.000000    708.000000      1.000000   
50%         103.000000        17.000000   1088.000000      1.000000   
75%         137.000000        24.000000   1650.000000      3.000000   
max         860.000000       127.000000  15840.000000    170.000000   

       command_count  control_point_proxy  primitive_type_count  
count   49999.000000         49999.000000          49999.000000  
mean       46.904578            53.483650              1.085662  
std        34.751539            48.509471              0.351350  
min         0.000000             0.000000              0.000000  
25%        25.000000          

,id,prompt,svg_len,path_count,command_count,control_point_proxy,primitive_type_count,source_group
33552,bdcb4c0f87a075134b991bb5c9ac0dec,A black icon featuring a vertical rectangle wi...,1344,1,63.0,56.0,1,main_like
9427,73d39ee0485b86ecc66ccc4875a0c2d6,A single black outline of a heart shape agains...,734,1,22.0,34.0,1,main_like
199,6c8fa0aa2a0303827ce4d98f7c95f3db,The image features a black-and-white geometric...,686,1,42.0,0.0,1,main_like
12447,eb2afeb60737c44c531b35e23b9c4f8c,A simple line graph with a rising trend depict...,438,1,20.0,0.0,1,main_like
39488,30b73e267fa71b3456d7aacb41e3d7d0,"The image features a simple, monochromatic dra...",2336,3,58.0,150.0,1,main_like



## Retrieval fallback

This is not the main model. It is a safety net.

If generation is invalid after sanitization, we can fall back to the most similar training prompt.
That usually scores better than returning an empty SVG.


In [20]:

# Chunk 10 — retrieval helper

class SVGNearestNeighbor:
    def __init__(self) -> None:
        self.vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=1)
        self.prompt_matrix = None
        self.prompts: List[str] = []
        self.svgs: List[str] = []

    def fit(self, prompts: Sequence[str], svgs: Sequence[str]) -> None:
        self.prompts = list(prompts)
        self.svgs = list(svgs)
        self.prompt_matrix = self.vectorizer.fit_transform(self.prompts)

    def retrieve(self, prompt: str) -> Tuple[str, float, int]:
        q = self.vectorizer.transform([prompt])
        sims = linear_kernel(q, self.prompt_matrix)[0]
        idx = int(np.argmax(sims))
        return self.svgs[idx], float(sims[idx]), idx


In [21]:
# Chunk 11 — quick 20% holdout + retrieval banks (scheme A)
# 目标：
# 1) 保留 leak-free 的 quick holdout / retrieval bank
# 2) direct run 时，训练源内部再切一个小的 train/valid split 用于 eval_loss
# 3) 继续避免 submission 与本地验证之间的数据泄漏

def safe_qcut_str(x: pd.Series, q: int, prefix: str) -> pd.Series:
    x = x.fillna(0)
    try:
        bins = pd.qcut(x, q=q, duplicates="drop")
        return bins.astype(str).radd(prefix + ":")
    except Exception:
        uniq = max(2, min(q, int(x.nunique())))
        bins = pd.cut(x.rank(method="first"), bins=uniq, duplicates="drop")
        return bins.astype(str).radd(prefix + ":")

def build_stratify_key(df: pd.DataFrame) -> pd.Series:
    work = df.copy()
    if "svg_len" not in work.columns or "command_count" not in work.columns:
        work = add_svg_stats(work)

    if "prompt_word_len" not in work.columns:
        work["prompt_word_len"] = work["prompt"].astype(str).str.split().str.len()
    if "is_external_like" not in work.columns:
        work["is_external_like"] = False

    work["source_group"] = np.where(work["is_external_like"], "external", "main")
    parts = [
        safe_qcut_str(work["svg_len"], q=6, prefix="svg"),
        safe_qcut_str(work["path_count"], q=5, prefix="path"),
        safe_qcut_str(work["command_count"], q=5, prefix="cmd"),
        safe_qcut_str(work["control_point_proxy"], q=5, prefix="ctrl"),
        safe_qcut_str(work["prompt_word_len"], q=5, prefix="prompt"),
        work["source_group"].astype(str).radd("src:"),
    ]
    key = (
        parts[0].astype(str) + "|" + parts[1].astype(str) + "|" + parts[2].astype(str)
        + "|" + parts[3].astype(str) + "|" + parts[4].astype(str) + "|" + parts[5].astype(str)
    )
    vc = key.value_counts()
    rare = vc[vc < 2].index
    key = key.where(~key.isin(rare), "fallback")
    return key

split_stratify_key = build_stratify_key(train_df)

quick_train_split, quick_holdout_split = train_test_split(
    train_df,
    test_size=CFG.QUICK_HOLDOUT_SIZE,
    random_state=CFG.SEED,
    shuffle=True,
    stratify=split_stratify_key,
)

if CFG.DEBUG_MODE:
    quick_train_split = maybe_limit(quick_train_split, CFG.DEBUG_TRAIN_SAMPLES, CFG.SEED)
    if CFG.DEBUG_QUICK_HOLDOUT_SAMPLES is not None:
        quick_holdout_split = maybe_limit(quick_holdout_split, CFG.DEBUG_QUICK_HOLDOUT_SAMPLES, CFG.SEED)

quick_train_split = quick_train_split.reset_index(drop=True)
quick_holdout_split = quick_holdout_split.reset_index(drop=True)

print("quick train split   :", len(quick_train_split))
print("quick holdout split :", len(quick_holdout_split))

quick_holdout_ids = set(quick_holdout_split["id"].astype(str).tolist())

train_pool_full = train_df[~train_df["id"].astype(str).isin(quick_holdout_ids)].copy().reset_index(drop=True)
train_pool_filtered = filtered_train_df[~filtered_train_df["id"].astype(str).isin(quick_holdout_ids)].copy().reset_index(drop=True)
train_pool_split = quick_train_split.copy().reset_index(drop=True)

submission_pool_full = train_df.copy().reset_index(drop=True)
submission_pool_filtered = filtered_train_df.copy().reset_index(drop=True)
submission_pool_split = quick_train_split.copy().reset_index(drop=True)

print("quick-valid pool [full]    :", len(train_pool_full))
print("quick-valid pool [filtered]:", len(train_pool_filtered))
print("quick-valid pool [split]   :", len(train_pool_split))
print("submission pool [full]     :", len(submission_pool_full))
print("submission pool [filtered] :", len(submission_pool_filtered))
print("submission pool [split]    :", len(submission_pool_split))

def _build_nn_index(df: pd.DataFrame) -> SVGNearestNeighbor:
    nn = SVGNearestNeighbor()
    nn.fit(df["prompt"].astype(str).tolist(), df["svg"].astype(str).tolist())
    return nn

def get_active_valid_retrieval_index_and_bank():
    src = CFG.RETRIEVAL_SOURCE
    if src == "full":
        bank = train_pool_full
    elif src == "filtered":
        bank = train_pool_filtered
    elif src == "split":
        bank = train_pool_split
    else:
        raise ValueError(f"Unknown RETRIEVAL_SOURCE: {src}")
    return _build_nn_index(bank), bank

def get_active_submission_retrieval_index_and_bank():
    src = CFG.RETRIEVAL_SOURCE
    if src == "full":
        bank = submission_pool_full
    elif src == "filtered":
        bank = submission_pool_filtered
    elif src == "split":
        bank = submission_pool_split
    else:
        raise ValueError(f"Unknown RETRIEVAL_SOURCE: {src}")
    return _build_nn_index(bank), bank

def get_lora_train_df():
    if CFG.LORA_TRAIN_SOURCE == "full":
        return train_pool_full
    elif CFG.LORA_TRAIN_SOURCE == "filtered":
        return train_pool_filtered
    elif CFG.LORA_TRAIN_SOURCE == "split":
        return train_pool_split
    raise ValueError(f"Unknown LORA_TRAIN_SOURCE: {CFG.LORA_TRAIN_SOURCE}")

def get_lora_train_valid_split() -> Tuple[pd.DataFrame, Optional[pd.DataFrame]]:
    source_df = get_lora_train_df().copy().reset_index(drop=True)

    if (
        not CFG.USE_TRAIN_VALIDATION
        or CFG.TRAIN_VALID_SIZE <= 0
        or len(source_df) < max(CFG.TRAIN_VALID_MIN_ROWS, 32)
    ):
        return source_df.reset_index(drop=True), None

    strat_key = build_stratify_key(source_df)
    train_part, valid_part = train_test_split(
        source_df,
        test_size=CFG.TRAIN_VALID_SIZE,
        random_state=CFG.SEED,
        shuffle=True,
        stratify=strat_key,
    )

    train_part = train_part.reset_index(drop=True)
    valid_part = valid_part.reset_index(drop=True)

    if CFG.DEBUG_MODE:
        train_part = maybe_limit(train_part, CFG.DEBUG_TRAIN_SAMPLES, CFG.SEED)
        if CFG.DEBUG_VALID_SAMPLES is not None:
            valid_part = maybe_limit(valid_part, CFG.DEBUG_VALID_SAMPLES, CFG.SEED)

    if CFG.TRAIN_VALID_MAX_ROWS is not None:
        valid_part = maybe_limit(valid_part, CFG.TRAIN_VALID_MAX_ROWS, CFG.SEED)

    return train_part.reset_index(drop=True), valid_part.reset_index(drop=True)

active_valid_nn_index, active_valid_retrieval_bank = get_active_valid_retrieval_index_and_bank()
active_submission_nn_index, active_submission_retrieval_bank = get_active_submission_retrieval_index_and_bank()

train_split = quick_train_split.copy()
valid_split = quick_holdout_split.copy()
nn_index = active_submission_nn_index
nn_bank = active_submission_retrieval_bank

print("ACTIVE retrieval source       :", CFG.RETRIEVAL_SOURCE)
print("ACTIVE quick-valid bank size  :", len(active_valid_retrieval_bank))
print("ACTIVE submission bank size   :", len(active_submission_retrieval_bank))
print("LoRA train source             :", CFG.LORA_TRAIN_SOURCE)

retrieval_valid_preds = []
retrieval_valid_sims = []

for _, row in quick_holdout_split.iterrows():
    nn_svg, sim, _ = active_valid_nn_index.retrieve(str(row["prompt"]))
    retrieval_valid_preds.append(nn_svg)
    retrieval_valid_sims.append(sim)

valid_eval_df = quick_holdout_split.copy().reset_index(drop=True)
valid_eval_df["retrieval_svg"] = retrieval_valid_preds
valid_eval_df["retrieval_sim"] = retrieval_valid_sims

quick train split   : 39999
quick holdout split : 10000
quick-valid pool [full]    : 39999
quick-valid pool [filtered]: 26387
quick-valid pool [split]   : 39999
submission pool [full]     : 49999
submission pool [filtered] : 32954
submission pool [split]    : 39999
ACTIVE retrieval source       : full
ACTIVE quick-valid bank size  : 39999
ACTIVE submission bank size   : 49999
LoRA train source             : filtered


In [22]:
# Chunk 11.1 — override retrieval / training pools with curriculum-aware sources

def _exclude_holdout(df: pd.DataFrame) -> pd.DataFrame:
    return df[~df["id"].astype(str).isin(quick_holdout_ids)].copy().reset_index(drop=True)

train_pool_by_source = {
    "full": _exclude_holdout(train_df),
    "filtered": _exclude_holdout(filtered_train_df),
    "filtered_easy": _exclude_holdout(filtered_easy_df),
    "filtered_medium": _exclude_holdout(filtered_medium_df),
    "filtered_hard": _exclude_holdout(filtered_hard_df),
    "curriculum_stage_a": _exclude_holdout(curriculum_stage_a_df),
    "curriculum_stage_b": _exclude_holdout(curriculum_stage_b_df),
    "hard_focus": _exclude_holdout(hard_focus_df),
    "split": quick_train_split.copy().reset_index(drop=True),
}

submission_pool_by_source = {
    "full": train_df.copy().reset_index(drop=True),
    "filtered": filtered_train_df.copy().reset_index(drop=True),
    "filtered_easy": filtered_easy_df.copy().reset_index(drop=True),
    "filtered_medium": filtered_medium_df.copy().reset_index(drop=True),
    "filtered_hard": filtered_hard_df.copy().reset_index(drop=True),
    "curriculum_stage_a": curriculum_stage_a_df.copy().reset_index(drop=True),
    "curriculum_stage_b": curriculum_stage_b_df.copy().reset_index(drop=True),
    "hard_focus": hard_focus_df.copy().reset_index(drop=True),
    "split": quick_train_split.copy().reset_index(drop=True),
}

print("available training sources:", list(train_pool_by_source.keys()))
for k, v in train_pool_by_source.items():
    print(f"train_pool[{k:>18}] =", len(v))
for k, v in submission_pool_by_source.items():
    print(f"submission_pool[{k:>13}] =", len(v))

def get_active_valid_retrieval_index_and_bank():
    src = CFG.RETRIEVAL_SOURCE
    if src not in train_pool_by_source:
        raise ValueError(f"Unknown RETRIEVAL_SOURCE: {src}")
    bank = train_pool_by_source[src]
    return _build_nn_index(bank), bank

def get_active_submission_retrieval_index_and_bank():
    src = CFG.RETRIEVAL_SOURCE
    if src not in submission_pool_by_source:
        raise ValueError(f"Unknown RETRIEVAL_SOURCE: {src}")
    bank = submission_pool_by_source[src]
    return _build_nn_index(bank), bank

def get_lora_train_df():
    src = CFG.LORA_TRAIN_SOURCE
    if src not in train_pool_by_source:
        raise ValueError(f"Unknown LORA_TRAIN_SOURCE: {src}")
    return train_pool_by_source[src]

active_valid_nn_index, active_valid_retrieval_bank = get_active_valid_retrieval_index_and_bank()
active_submission_nn_index, active_submission_retrieval_bank = get_active_submission_retrieval_index_and_bank()

print("OVERRIDE active retrieval source :", CFG.RETRIEVAL_SOURCE)
print("OVERRIDE active LoRA source      :", CFG.LORA_TRAIN_SOURCE)
print("OVERRIDE active valid bank size  :", len(active_valid_retrieval_bank))
print("OVERRIDE active submit bank size :", len(active_submission_retrieval_bank))

available training sources: ['full', 'filtered', 'filtered_easy', 'filtered_medium', 'filtered_hard', 'curriculum_stage_a', 'curriculum_stage_b', 'hard_focus', 'split']
train_pool[              full] = 39999
train_pool[          filtered] = 26387
train_pool[     filtered_easy] = 26387
train_pool[   filtered_medium] = 32085
train_pool[     filtered_hard] = 35613
train_pool[curriculum_stage_a] = 32085
train_pool[curriculum_stage_b] = 35613
train_pool[        hard_focus] = 29389
train_pool[             split] = 39999
submission_pool[         full] = 49999
submission_pool[     filtered] = 32954
submission_pool[filtered_easy] = 32954
submission_pool[filtered_medium] = 40095
submission_pool[filtered_hard] = 44513
submission_pool[curriculum_stage_a] = 40095
submission_pool[curriculum_stage_b] = 44513
submission_pool[   hard_focus] = 36688
submission_pool[        split] = 39999
OVERRIDE active retrieval source : full
OVERRIDE active LoRA source      : filtered
OVERRIDE active valid bank size  


## Dataset and collator

Important detail: for supervised fine-tuning, we mask the **prompt part** of the sequence with `-100`,
so loss is only computed on the assistant SVG tokens.


In [23]:

# Chunk 11.5 — local proxy metric for V / S / C on validation
# 不是官方指标，只做方向判断。
# 这里优先用 skimage；如果环境里没有装，会自动退回 numpy fallback，不会直接报错。

import io
import math
import numpy as np
import pandas as pd
import xml.etree.ElementTree as ET

from PIL import Image
import cairosvg
import zss

try:
    from skimage.metrics import structural_similarity as skimage_ssim
    from skimage.feature import canny as skimage_canny
    HAS_SKIMAGE = True
except Exception:
    HAS_SKIMAGE = False
    skimage_ssim = None
    skimage_canny = None

print("HAS_SKIMAGE:", HAS_SKIMAGE)

# -----------------------------
# helpers
# -----------------------------
def local_name_safe(tag: str) -> str:
    if tag is None:
        return ""
    if "}" in tag:
        return tag.split("}", 1)[1]
    return tag

def render_svg_to_gray(svg_text: str, size: int = 256) -> np.ndarray:
    """
    Render SVG to 256x256 grayscale image.
    White background, uint8 output in [0,255].
    Invalid SVG -> blank white canvas.
    """
    try:
        png_bytes = cairosvg.svg2png(
            bytestring=svg_text.encode("utf-8"),
            output_width=size,
            output_height=size,
            background_color="white",
        )
        img = Image.open(io.BytesIO(png_bytes)).convert("L")
        return np.array(img, dtype=np.uint8)
    except Exception:
        return np.full((size, size), 255, dtype=np.uint8)

def fallback_edge_mask(img: np.ndarray, threshold: float = 0.08) -> np.ndarray:
    arr = img.astype(np.float32) / 255.0
    gx = np.zeros_like(arr)
    gy = np.zeros_like(arr)
    gx[:, 1:] = np.abs(arr[:, 1:] - arr[:, :-1])
    gy[1:, :] = np.abs(arr[1:, :] - arr[:-1, :])
    grad = np.sqrt(gx * gx + gy * gy)
    return grad > threshold

def edge_f1_score(img_pred: np.ndarray, img_true: np.ndarray) -> float:
    """
    Edge-F1 proxy using skimage.canny if available, otherwise numpy gradient fallback.
    """
    pred_norm = img_pred.astype(np.float32) / 255.0
    true_norm = img_true.astype(np.float32) / 255.0

    if HAS_SKIMAGE:
        edge_pred = skimage_canny(pred_norm)
        edge_true = skimage_canny(true_norm)
    else:
        edge_pred = fallback_edge_mask(img_pred)
        edge_true = fallback_edge_mask(img_true)

    tp = np.logical_and(edge_pred, edge_true).sum()
    fp = np.logical_and(edge_pred, ~edge_true).sum()
    fn = np.logical_and(~edge_pred, edge_true).sum()

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    if precision + recall == 0:
        return 0.0
    return float(2 * precision * recall / (precision + recall + 1e-8))

def fallback_ssim(img_true: np.ndarray, img_pred: np.ndarray) -> float:
    """
    Very rough SSIM-like fallback if skimage is unavailable.
    Not equivalent to official SSIM, only used to keep local validation running.
    """
    x = img_true.astype(np.float32) / 255.0
    y = img_pred.astype(np.float32) / 255.0

    mu_x = x.mean()
    mu_y = y.mean()
    var_x = x.var()
    var_y = y.var()
    cov_xy = ((x - mu_x) * (y - mu_y)).mean()

    c1 = 0.01 ** 2
    c2 = 0.03 ** 2

    num = (2 * mu_x * mu_y + c1) * (2 * cov_xy + c2)
    den = (mu_x ** 2 + mu_y ** 2 + c1) * (var_x + var_y + c2)
    val = num / (den + 1e-8)
    return float(np.clip(val, 0.0, 1.0))

def visual_score(svg_pred: str, svg_true: str) -> float:
    """
    V proxy = mean(SSIM, Edge-F1)
    """
    img_pred = render_svg_to_gray(svg_pred, size=256)
    img_true = render_svg_to_gray(svg_true, size=256)

    if HAS_SKIMAGE:
        ssim_val = skimage_ssim(img_true, img_pred, data_range=255)
    else:
        ssim_val = fallback_ssim(img_true, img_pred)

    edge_val = edge_f1_score(img_pred, img_true)
    return float((ssim_val + edge_val) / 2.0)

def node_label(el: ET.Element) -> str:
    tag = local_name_safe(el.tag)
    attr_keys = ",".join(sorted(el.attrib.keys()))
    return f"{tag}|{attr_keys}"

def build_zss_tree(el: ET.Element) -> zss.Node:
    root = zss.Node(node_label(el))
    for child in list(el):
        root.addkid(build_zss_tree(child))
    return root

def structural_score(svg_pred: str, svg_true: str) -> float:
    """
    S proxy = exp(-TED / 25)
    """
    try:
        root_pred = ET.fromstring(svg_pred)
        root_true = ET.fromstring(svg_true)
    except Exception:
        return 0.0

    tree_pred = build_zss_tree(root_pred)
    tree_true = build_zss_tree(root_true)

    ted = zss.simple_distance(tree_pred, tree_true)
    return float(math.exp(-ted / 25.0))

def compactness_score(svg_pred: str, svg_true: str) -> float:
    lp = len(svg_pred)
    lt = len(svg_true)
    ratio = (lp + 1) / (lt + 1)
    return float(math.exp(-abs(math.log(ratio))))

def pair_proxy_score(svg_pred: str, svg_true: str) -> dict:
    if not isinstance(svg_pred, str) or not isinstance(svg_true, str):
        return {"V": 0.0, "S": 0.0, "C": 0.0, "score": 0.0}

    if not is_valid_svg(svg_pred):
        return {"V": 0.0, "S": 0.0, "C": 0.0, "score": 0.0}

    V = visual_score(svg_pred, svg_true)
    S = structural_score(svg_pred, svg_true)
    C = compactness_score(svg_pred, svg_true)

    eps = 1e-8
    score = math.exp(
        0.85 * math.log(max(V, eps)) +
        0.12 * math.log(max(S, eps)) +
        0.03 * math.log(max(C, eps))
    )

    return {"V": V, "S": S, "C": C, "score": score}

def evaluate_valid_predictions(valid_df: pd.DataFrame, pred_col: str, gt_col: str = "svg") -> pd.DataFrame:
    rows = []
    for _, row in valid_df.iterrows():
        rows.append(pair_proxy_score(row[pred_col], row[gt_col]))
    score_df = pd.DataFrame(rows)
    out = valid_df.reset_index(drop=True).copy()
    out = pd.concat([out, score_df], axis=1)
    return out

def print_proxy_summary(eval_df: pd.DataFrame, name: str = "model") -> None:
    print(f"\n===== {name} proxy metric summary =====")
    print("rows:", len(eval_df))
    print("valid rate:", (eval_df["score"] > 0).mean())
    print("V mean:", eval_df["V"].mean())
    print("S mean:", eval_df["S"].mean())
    print("C mean:", eval_df["C"].mean())
    print("score mean:", eval_df["score"].mean())
    print("score median:", eval_df["score"].median())


HAS_SKIMAGE: True


In [24]:

# Chunk 12 — dataset and collator
def build_generation_user_content(prompt: str) -> str:
    return (
        f"{prompt}\n"
        "Return exactly one valid SVG. "
        "Do not use markdown fences. "
        "Do not explain anything. "
        "Start directly with <svg and end with </svg>. "
        "Use a 256x256 canvas. "
        "Prefer compact SVG."
    )

class SVGSFTDataset(torch.utils.data.Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, max_seq_len: int, system_prompt: Optional[str] = None) -> None:
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len
        self.system_prompt = system_prompt or build_system_prompt(CFG.TAG_MODE)

    def __len__(self) -> int:
        return len(self.df)

    def _build_prompt_prefix(self, prompt: str) -> str:
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": build_generation_user_content(prompt)},
        ]
        return self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    def __getitem__(self, idx: int) -> Dict[str, List[int]]:
        row = self.df.iloc[idx]
        prompt = str(row["prompt"])
        svg = str(row["svg"])

        prefix = self._build_prompt_prefix(prompt)
        full_text = prefix + svg + (self.tokenizer.eos_token or "")

        prefix_ids = self.tokenizer(prefix, add_special_tokens=False)["input_ids"]
        tokenized = self.tokenizer(
            full_text,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_seq_len,
        )

        input_ids = tokenized["input_ids"]
        attention_mask = tokenized["attention_mask"]
        labels = input_ids.copy()

        prompt_len = min(len(prefix_ids), len(labels))
        labels[:prompt_len] = [-100] * prompt_len

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }


class CausalLMCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.pad_id = tokenizer.pad_token_id

    def __call__(self, features: List[Dict[str, List[int]]]) -> Dict[str, torch.Tensor]:
        max_len = max(len(f["input_ids"]) for f in features)
        batch_input_ids, batch_attention_mask, batch_labels = [], [], []

        for f in features:
            pad_len = max_len - len(f["input_ids"])
            batch_input_ids.append(f["input_ids"] + [self.pad_id] * pad_len)
            batch_attention_mask.append(f["attention_mask"] + [0] * pad_len)
            batch_labels.append(f["labels"] + [-100] * pad_len)

        return {
            "input_ids": torch.tensor(batch_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(batch_attention_mask, dtype=torch.long),
            "labels": torch.tensor(batch_labels, dtype=torch.long),
        }




## Model loading

For this task, a coder model is usually a better default than a generic chat model,
because the output is tightly structured code-like text.


In [25]:

# Chunk 13 — model and tokenizer loaders

def _preferred_torch_dtype():
    if CFG.BF16:
        return torch.bfloat16
    return torch.float16

def load_tokenizer(base_model_path: str):
    tokenizer = AutoTokenizer.from_pretrained(base_model_path, trust_remote_code=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    return tokenizer

def maybe_prepare_adapter_dir_for_training() -> None:
    """
    If an adapter dir already exists but belongs to another base model / LoRA config,
    archive it before retraining so we don't accidentally overwrite or later reload a mismatched adapter.
    """
    if not os.path.isdir(CFG.ADAPTER_DIR):
        ensure_dir(CFG.ADAPTER_DIR)
        return

    existing_meta = load_adapter_meta(CFG.ADAPTER_DIR)

    # 没有 meta：旧版 adapter，保险起见也归档
    if existing_meta is None:
        if CFG.AUTO_ARCHIVE_MISMATCHED_ADAPTER:
            archived = archive_dir(CFG.ADAPTER_DIR)
            print(f"[INFO] Existing adapter had no metadata. Archived to: {archived}")
            ensure_dir(CFG.ADAPTER_DIR)
        return

    if not adapter_meta_matches_cfg(existing_meta):
        if CFG.AUTO_ARCHIVE_MISMATCHED_ADAPTER:
            archived = archive_dir(CFG.ADAPTER_DIR)
            print(f"[INFO] Archived mismatched adapter to: {archived}")
            print("[INFO] Will retrain a fresh adapter for current CFG.")
            ensure_dir(CFG.ADAPTER_DIR)
        else:
            raise RuntimeError(
                "Adapter dir exists but metadata does not match current CFG. "
                "Enable AUTO_ARCHIVE_MISMATCHED_ADAPTER or change ADAPTER_DIR."
            )


def load_base_model_for_lora(base_model_path: str):
    if CFG.USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=_preferred_torch_dtype(),
            bnb_4bit_use_double_quant=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            trust_remote_code=False,
            device_map="auto",
            quantization_config=bnb_config,
            torch_dtype=_preferred_torch_dtype(),
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            trust_remote_code=False,
            device_map="auto",
            torch_dtype=_preferred_torch_dtype(),
        )

    model.config.use_cache = False

    lora_config = LoraConfig(
        r=CFG.LORA_R,
        lora_alpha=CFG.LORA_ALPHA,
        lora_dropout=CFG.LORA_DROPOUT,
        target_modules=list(CFG.TARGET_MODULES),
        bias="none",
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(model, lora_config)
    return model

def print_trainable_parameters(model) -> None:
    trainable_params = 0
    total_params = 0
    for _, p in model.named_parameters():
        total_params += p.numel()
        if p.requires_grad:
            trainable_params += p.numel()
    pct = 100 * trainable_params / total_params
    print(f"Trainable params: {trainable_params:,} / {total_params:,} ({pct:.4f}%)")

def load_model_for_inference():
    tokenizer = load_tokenizer(CFG.BASE_MODEL_PATH)

    adapter_cfg_path = os.path.join(CFG.ADAPTER_DIR, "adapter_config.json")
    adapter_exists = os.path.isdir(CFG.ADAPTER_DIR) and os.path.exists(adapter_cfg_path)
    adapter_meta = load_adapter_meta(CFG.ADAPTER_DIR)
    adapter_match = adapter_meta_matches_cfg(adapter_meta)

    use_adapter = CFG.LOAD_EXISTING_ADAPTER and adapter_exists

    if use_adapter and CFG.STRICT_ADAPTER_MATCH and not adapter_match:
        print("[WARN] Adapter metadata does not match current CFG / base model.")
        print("[WARN] Current base model:", CFG.BASE_MODEL_PATH)
        print("[WARN] Adapter meta      :", adapter_meta)
        if CFG.ALLOW_BASE_ONLY_INFERENCE_IF_ADAPTER_MISMATCH:
            print("[WARN] Falling back to base model only.")
            use_adapter = False
        else:
            raise RuntimeError(
                "Adapter exists but does not match current base model. "
                "Set LOAD_EXISTING_ADAPTER=False or retrain a matching adapter."
            )

    if use_adapter:
        print(f"[INFO] Loading base model from: {CFG.BASE_MODEL_PATH}")
        print(f"[INFO] Loading adapter from: {CFG.ADAPTER_DIR}")
        base_model = AutoModelForCausalLM.from_pretrained(
            CFG.BASE_MODEL_PATH,
            trust_remote_code=False,
            device_map="auto",
            dtype=_preferred_torch_dtype(),
        )
        model = PeftModel.from_pretrained(base_model, CFG.ADAPTER_DIR)
        if CFG.MERGE_ADAPTER_FOR_INFERENCE and hasattr(model, "merge_and_unload"):
            print("[INFO] merge_and_unload() for inference ...")
            model = model.merge_and_unload()
    else:
        print(f"[INFO] Loading base model only from: {CFG.BASE_MODEL_PATH}")
        model = AutoModelForCausalLM.from_pretrained(
            CFG.BASE_MODEL_PATH,
            trust_remote_code=False,
            device_map="auto",
            dtype=_preferred_torch_dtype(),
        )

    model.eval()
    return model, tokenizer


In [26]:

# Chunk 13.1 — token length diagnostics (why MAX_SEQ_LEN matters)
# 用一小部分样本估计：当前 chat template + SVG 在 tokenizer 下会占多少 token。
# 这格不是训练必须，但它能直接帮助判断 768 / 1536 / 2048 哪个更合理。

_tokenizer_for_diag = load_tokenizer(CFG.BASE_MODEL_PATH)
diag_sample_n = min(512, len(train_df))
diag_df = train_df.sample(diag_sample_n, random_state=CFG.SEED).reset_index(drop=True)

def estimate_token_lengths(df: pd.DataFrame, tokenizer, system_prompt: Optional[str] = None) -> pd.DataFrame:
    system_prompt = system_prompt or build_system_prompt(CFG.TAG_MODE)
    rows = []
    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        svg = str(row["svg"])
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt},
        ]
        prefix = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        prefix_len = len(tokenizer(prefix, add_special_tokens=False)["input_ids"])
        svg_len = len(tokenizer(svg, add_special_tokens=False)["input_ids"])
        full_len = len(tokenizer(prefix + svg + (_tokenizer_for_diag.eos_token or ""), add_special_tokens=False)["input_ids"])
        rows.append({
            "prompt_chars": len(prompt),
            "svg_chars": len(svg),
            "prefix_tokens": prefix_len,
            "svg_tokens": svg_len,
            "full_tokens": full_len,
        })
    return pd.DataFrame(rows)

token_diag_df = estimate_token_lengths(diag_df, _tokenizer_for_diag)
display(token_diag_df.describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T)

for seq_len in [768, 1024, 1536, 2048]:
    trunc_rate = (token_diag_df["full_tokens"] > seq_len).mean()
    print(f"Estimated truncation rate at seq_len={seq_len}: {trunc_rate:.2%}")

print("Current CFG.MAX_SEQ_LEN:", CFG.MAX_SEQ_LEN)


,count,mean,std,min,50%,75%,90%,95%,99%,max
prompt_chars,512.0,112.609375,59.481943,32.0,102.0,133.00,177.9,238.35,322.00,404.0
svg_chars,512.0,1272.898438,798.989250,238.0,1088.0,1649.75,2300.1,2695.40,3747.99,6530.0
prefix_tokens,512.0,218.724609,10.786867,204.0,216.0,223.00,231.0,241.45,258.78,268.0
svg_tokens,512.0,1066.871094,732.562008,136.0,885.5,1430.00,1994.5,2346.90,3390.96,5939.0
full_tokens,512.0,1286.595703,732.743032,347.0,1102.5,1652.25,2206.1,2578.80,3610.86,6154.0


Estimated truncation rate at seq_len=768: 73.24%
Estimated truncation rate at seq_len=1024: 55.08%
Estimated truncation rate at seq_len=1536: 29.49%
Estimated truncation rate at seq_len=2048: 14.45%
Current CFG.MAX_SEQ_LEN: 2048


## Training

This version keeps the direct-run training path, but:

1. uses a small **train/valid split** for the chosen LoRA source when `USE_TRAIN_VALIDATION = True`
2. saves `trainer_log_history.csv` and `eval_history.csv` next to the adapter
3. can load the best checkpoint by `eval_loss` during the single-run training flow

In [27]:

# Chunk 14.1 — training function used by the unified scheduler

def load_raw_base_model_for_training(base_model_path: str):
    if CFG.USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=_preferred_torch_dtype(),
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            trust_remote_code=False,
            device_map="auto",
            quantization_config=bnb_config,
            torch_dtype=_preferred_torch_dtype(),
        )
        model = prepare_model_for_kbit_training(model)
    else:
        model = AutoModelForCausalLM.from_pretrained(
            base_model_path,
            trust_remote_code=False,
            device_map="auto",
            torch_dtype=_preferred_torch_dtype(),
        )
    model.config.use_cache = False
    return model

def train_lora_model(
    train_df: pd.DataFrame,
    valid_df: Optional[pd.DataFrame] = None,
    run_name: Optional[str] = None,
):
    maybe_prepare_adapter_dir_for_training()

    tokenizer = load_tokenizer(CFG.BASE_MODEL_PATH)
    init_adapter_dir = getattr(CFG, "INIT_ADAPTER_DIR", None)

    has_init_adapter = (
        isinstance(init_adapter_dir, str)
        and len(init_adapter_dir.strip()) > 0
        and os.path.isdir(init_adapter_dir)
        and os.path.exists(os.path.join(init_adapter_dir, "adapter_config.json"))
    )

    if has_init_adapter:
        print("[INFO] Continue LoRA training from adapter:", init_adapter_dir)
        base_model = load_raw_base_model_for_training(CFG.BASE_MODEL_PATH)
        model = PeftModel.from_pretrained(base_model, init_adapter_dir, is_trainable=True)
        model.config.use_cache = False
    else:
        model = load_base_model_for_lora(CFG.BASE_MODEL_PATH)

    print_trainable_parameters(model)

    train_dataset = SVGSFTDataset(train_df, tokenizer, CFG.MAX_SEQ_LEN)
    collator = CausalLMCollator(tokenizer)

    has_valid = valid_df is not None and len(valid_df) > 0
    valid_dataset = SVGSFTDataset(valid_df, tokenizer, CFG.MAX_SEQ_LEN) if has_valid else None

    args_kwargs = dict(
        output_dir=CFG.OUTPUT_DIR,
        num_train_epochs=CFG.NUM_EPOCHS,
        per_device_train_batch_size=CFG.TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=CFG.EVAL_BATCH_SIZE,
        gradient_accumulation_steps=CFG.GRAD_ACCUM_STEPS,
        learning_rate=CFG.LEARNING_RATE,
        weight_decay=CFG.WEIGHT_DECAY,
        warmup_steps=CFG.WARMUP_STEPS,
        logging_steps=CFG.LOGGING_STEPS,
        save_strategy="steps",
        save_steps=CFG.SAVE_STEPS,
        bf16=bool(CFG.BF16),
        fp16=not bool(CFG.BF16),
        lr_scheduler_type="cosine",
        report_to="none",
        save_total_limit=2,
        gradient_checkpointing=CFG.GRADIENT_CHECKPOINTING,
        dataloader_num_workers=0,
        remove_unused_columns=False,
        max_grad_norm=0.3,
        optim=CFG.TRAIN_OPTIM,
        seed=CFG.SEED,
        do_train=True,
        do_eval=bool(has_valid),
    )

    if has_valid:
        args_kwargs.update(
            eval_strategy="steps",
            eval_steps=CFG.EVAL_STEPS,
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        )
    else:
        args_kwargs.update(
            eval_strategy="no",
            load_best_model_at_end=False,
        )

    args = TrainingArguments(**args_kwargs)

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_dataset,
        eval_dataset=valid_dataset,
        data_collator=collator,
    )

    trainer.train()
    trainer.model.save_pretrained(CFG.ADAPTER_DIR)
    tokenizer.save_pretrained(CFG.ADAPTER_DIR)
    save_adapter_meta(CFG.ADAPTER_DIR)

    history_df = pd.DataFrame(trainer.state.log_history)
    history_path = os.path.join(CFG.ADAPTER_DIR, "trainer_log_history.csv")
    history_df.to_csv(history_path, index=False)

    if has_valid and "eval_loss" in history_df.columns:
        eval_history_df = history_df[history_df["eval_loss"].notna()].copy().reset_index(drop=True)
        eval_history_path = os.path.join(CFG.ADAPTER_DIR, "eval_history.csv")
        eval_history_df.to_csv(eval_history_path, index=False)
        if len(eval_history_df) > 0:
            print("[INFO] Validation checkpoints:")
            display(eval_history_df[["step", "epoch", "eval_loss"]].tail(10))

    print("[INFO] Saved adapter to:", CFG.ADAPTER_DIR)
    print("[INFO] Saved trainer history to:", history_path)
    return trainer.model, tokenizer, history_df


In [28]:

# Chunk 14 — deprecated duplicate training function removed
# 保留 14.1 作为唯一 train_lora_model 定义，避免后面的 cell 覆盖 continuation / unified scheduler 版本。
print("Chunk 14 skipped: using train_lora_model from Chunk 14.1")


Chunk 14 skipped: using train_lora_model from Chunk 14.1


In [29]:

# Chunk 15 — deprecated direct-run training removed
# 统一使用 Chunk 16.1 的训练调度器，避免 run-all 时重复训练两次。
print("Chunk 15 skipped: training is handled by Chunk 16.1")


Chunk 15 skipped: training is handled by Chunk 16.1



## Inference

During generation:
- first try the model output
- sanitize it
- if it is still invalid, use nearest-neighbor fallback
- if the prompt is almost identical to a training prompt, use direct retrieval immediately


In [30]:

# Chunk 16 — inference helpers

def build_inference_prompt(tokenizer, prompt: str) -> str:
    messages = [
        {"role": "system", "content": build_system_prompt(CFG.TAG_MODE)},
        {"role": "user", "content": build_generation_user_content(prompt)},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def generate_svg(model, tokenizer, prompt: str) -> str:
    text = build_inference_prompt(tokenizer, prompt)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    bad_words = ["```", "`"]
    bad_words_ids = []
    for bw in bad_words:
        ids = tokenizer(bw, add_special_tokens=False)["input_ids"]
        if len(ids) > 0:
            bad_words_ids.append(ids)

    generate_kwargs = dict(
        max_new_tokens=CFG.MAX_NEW_TOKENS,
        repetition_penalty=CFG.REPETITION_PENALTY,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        bad_words_ids=bad_words_ids if len(bad_words_ids) > 0 else None,
    )

    if CFG.DO_SAMPLE:
        generate_kwargs.update(
            do_sample=True,
            temperature=CFG.TEMPERATURE,
            top_p=CFG.TOP_P,
            num_beams=1,
        )
    else:
        generate_kwargs.update(
            do_sample=False,
            num_beams=CFG.NUM_BEAMS,
        )

    out = model.generate(**inputs, **generate_kwargs)
    generated_ids = out[0][inputs["input_ids"].shape[1]:]
    generated_text = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    generated_text = (
        generated_text
        .replace("```svg", "")
        .replace("```xml", "")
        .replace("```", "")
        .strip()
    )
    start = generated_text.lower().find("<svg")
    if start != -1:
        generated_text = generated_text[start:].strip()

    generated_text = trim_after_svg(generated_text)
    if "<svg" in generated_text.lower() and "</svg>" not in generated_text.lower():
        generated_text = repair_svg_candidate(generated_text)

    return generated_text.strip()

def maybe_sample_eval_df(df: pd.DataFrame, max_rows: Optional[int], seed: int) -> pd.DataFrame:
    if max_rows is None or len(df) <= max_rows:
        return df.copy().reset_index(drop=True)
    return df.sample(max_rows, random_state=seed).reset_index(drop=True)

def ensure_inference_model_loaded():
    global model, tokenizer
    if "model" not in globals() or "tokenizer" not in globals():
        model, tokenizer = load_model_for_inference()
    return model, tokenizer

def predict_single_prompt(
    prompt: str,
    nn_svg: str,
    sim: float,
    model=None,
    tokenizer=None,
) -> Dict[str, Any]:
    mode = CFG.SUBMISSION_MODE
    route = "retrieval"
    raw_pred = None
    sanitize_reason = "not_called"
    pred_svg = nn_svg
    raw_has_svg = False
    raw_has_close = False
    raw_len = 0

    need_generation = False
    if mode == "retrieval_only":
        need_generation = False
        route = "retrieval"
    elif mode == "hybrid":
        need_generation = sim < CFG.DIRECT_RETRIEVAL_THRESHOLD
        route = "retrieval" if not need_generation else "generated"
    elif mode == "generation_first":
        need_generation = True
        route = "generated"
    else:
        raise ValueError(f"Unknown submission mode: {mode}")

    if need_generation:
        if model is None or tokenizer is None:
            model, tokenizer = ensure_inference_model_loaded()

        raw_pred = generate_svg(model, tokenizer, prompt)
        raw_len = len(raw_pred)
        raw_has_svg = "<svg" in raw_pred.lower()
        raw_has_close = "</svg>" in raw_pred.lower()
        clean_svg, sanitize_reason = sanitize_svg_with_reason(
            raw_pred,
            max_chars=CFG.MAX_SVG_CHARS,
            sanitizer_mode=CFG.SANITIZER_MODE,
            tag_mode=CFG.TAG_MODE,
        )
        clean_retrieval = sanitize_svg(
            nn_svg,
            max_chars=CFG.MAX_SVG_CHARS,
            sanitizer_mode=CFG.SANITIZER_MODE,
            tag_mode=CFG.TAG_MODE,
        )
        if clean_retrieval is None:
            clean_retrieval = nn_svg

        if (
            clean_svg is not None
            and is_valid_svg(clean_svg, tag_mode=CFG.TAG_MODE)
            and has_visible_shapes(clean_svg, tag_mode=CFG.TAG_MODE)
        ):
            pred_svg = clean_svg
            route = "generated"
        else:
            pred_svg = clean_retrieval if is_valid_svg(clean_retrieval, tag_mode=CFG.TAG_MODE) else EMPTY_SVG
            route = "fallback"

    return {
        "pred_svg": pred_svg,
        "route": route,
        "sanitize_reason": sanitize_reason,
        "raw_pred": raw_pred,
        "raw_len": raw_len,
        "raw_has_svg": raw_has_svg,
        "raw_has_close": raw_has_close,
        "pred_valid": (
            is_valid_svg(pred_svg, tag_mode=CFG.TAG_MODE)
            and has_visible_shapes(pred_svg, tag_mode=CFG.TAG_MODE)
        ),
        "used_generation": need_generation,
    }

def run_prediction_pipeline(
    eval_df: pd.DataFrame,
    retrieval_index,
    run_name: str = "adhoc",
    max_rows: Optional[int] = None,
    save_debug_csv: bool = False,
):
    from tqdm.auto import tqdm

    work_df = maybe_sample_eval_df(eval_df, max_rows=max_rows, seed=CFG.SEED)
    work_df = work_df.reset_index(drop=True)

    need_generation = CFG.SUBMISSION_MODE in {"hybrid", "generation_first"}
    local_model, local_tokenizer = (None, None)
    if need_generation:
        local_model, local_tokenizer = ensure_inference_model_loaded()

    rows = []
    t0 = time.time()

    for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc=f"predict:{run_name}"):
        prompt = str(row["prompt"])
        nn_svg, sim, nn_idx = retrieval_index.retrieve(prompt)

        pred_info = predict_single_prompt(
            prompt=prompt,
            nn_svg=nn_svg,
            sim=sim,
            model=local_model,
            tokenizer=local_tokenizer,
        )

        rows.append({
            "id": row.get("id", None),
            "prompt": prompt,
            "svg": row.get("svg", None),
            "retrieval_svg": nn_svg,
            "retrieval_sim": sim,
            "retrieval_rank0_idx": nn_idx,
            **pred_info,
        })

    pred_df = pd.DataFrame(rows)
    elapsed = time.time() - t0

    if save_debug_csv and CFG.SAVE_DEBUG_CSV:
        debug_path = os.path.join(CFG.OUTPUT_DIR, f"debug_{run_name}.csv")
        pred_df.drop(columns=["raw_pred"], errors="ignore").to_csv(debug_path, index=False)
        print("saved debug csv:", debug_path)

    return pred_df, elapsed

def evaluate_prediction_run(pred_df: pd.DataFrame, run_name: str) -> Tuple[pd.DataFrame, Dict[str, Any]]:
    metric_df = evaluate_valid_predictions(
        pred_df.copy(),
        pred_col="pred_svg",
        gt_col="svg",
    )

    summary = {
        "run_name": run_name,
        "submission_mode": CFG.SUBMISSION_MODE,
        "retrieval_source": CFG.RETRIEVAL_SOURCE,
        "tag_mode": CFG.TAG_MODE,
        "sanitizer_mode": CFG.SANITIZER_MODE,
        "threshold": CFG.DIRECT_RETRIEVAL_THRESHOLD,
        "max_new_tokens": CFG.MAX_NEW_TOKENS,
        "valid_rate": float(pred_df["pred_valid"].mean()),
        "score_mean": float(metric_df["score"].mean()),
        "score_median": float(metric_df["score"].median()),
        "V_mean": float(metric_df["V"].mean()),
        "S_mean": float(metric_df["S"].mean()),
        "C_mean": float(metric_df["C"].mean()),
        "retrieval_count": int((pred_df["route"] == "retrieval").sum()),
        "generated_count": int((pred_df["route"] == "generated").sum()),
        "fallback_count": int((pred_df["route"] == "fallback").sum()),
        "generation_attempts": int(pred_df["used_generation"].sum()),
        "sanitize_fail_count": int(((pred_df["used_generation"]) & (pred_df["route"] == "fallback")).sum()),
    }
    return metric_df, summary


In [31]:

# Chunk 16.1 — unified training scheduler
# baseline notebook 默认 TRAIN_PLAN="single"；保留兼容字段以免后续 chunk 断掉。

canon_tag = "canon" if getattr(CFG, "ENABLE_CANONICALIZATION", False) else "nocanon"
plan_tag = getattr(CFG, "TRAIN_PLAN", "single")
run_tag = getattr(CFG, "RUN_TAG", "v1")

PIPELINE_ROOT = os.path.join(CFG.OUTPUT_DIR, f"nyu331_{CFG.EXPERIMENT_PRESET}_{plan_tag}_{canon_tag}_{run_tag}_pipeline")
PIPELINE_ADAPTER_ROOT = os.path.join(PIPELINE_ROOT, "adapters")
PIPELINE_TRAINER_ROOT = os.path.join(PIPELINE_ROOT, "trainer_outputs")
PIPELINE_SUBMISSION_ROOT = os.path.join(PIPELINE_ROOT, "submissions")

for _p in [PIPELINE_ROOT, PIPELINE_ADAPTER_ROOT, PIPELINE_TRAINER_ROOT, PIPELINE_SUBMISSION_ROOT]:
    ensure_dir(_p)

TRAINING_JOB_REGISTRY = {}
RUN_TO_TRAIN_JOB = {}
EXPERIMENT_RUN_REGISTRY = {}

def reset_runtime_model_cache(verbose: bool = True) -> None:
    for name in ["model", "tokenizer"]:
        if name in globals():
            try:
                del globals()[name]
            except Exception:
                pass
    gc.collect()
    if torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass
    if verbose:
        print("[INFO] Cleared model/tokenizer cache and CUDA memory.")

def epoch_tag(epoch_value: float) -> str:
    if abs(epoch_value - int(epoch_value)) < 1e-9:
        return f"e{int(epoch_value)}"
    return "e" + str(epoch_value).replace(".", "p")

def model_tag() -> str:
    return getattr(CFG, "MODEL_TAG", "0p5b")

def build_job_name(stage_label: str, source: str, epochs: float) -> str:
    return f"{stage_label}_{source}_seq{CFG.MAX_SEQ_LEN}_{epoch_tag(epochs)}_{model_tag()}"

def build_training_job(stage_label: str, source: str, epochs: float, init_adapter_dir: Optional[str] = None) -> Dict[str, Any]:
    job_name = build_job_name(stage_label, source, epochs)
    adapter_dir = os.path.join(PIPELINE_ADAPTER_ROOT, f"svg_lora_adapter_{job_name}")
    trainer_output_dir = os.path.join(PIPELINE_TRAINER_ROOT, job_name)
    return {
        "job_name": job_name,
        "stage_label": stage_label,
        "source": source,
        "epochs": float(epochs),
        "adapter_dir": adapter_dir,
        "trainer_output_dir": trainer_output_dir,
        "init_adapter_dir": init_adapter_dir,
        "overrides": {
            "MAX_SEQ_LEN": 2048,
            "MAX_NEW_TOKENS": 896,
            "LORA_TRAIN_SOURCE": source,
            "NUM_EPOCHS": float(epochs),
            "ADAPTER_DIR": adapter_dir,
            "OUTPUT_DIR": trainer_output_dir,
            "DO_TRAIN": True,
            "LOAD_EXISTING_ADAPTER": False,
            "USE_TRAIN_VALIDATION": True,
            "SUBMISSION_MODE": "hybrid",
            "DIRECT_RETRIEVAL_THRESHOLD": 1.00,
            "RETRIEVAL_SOURCE": "full",
            "TAG_MODE": "official",
            "SANITIZER_MODE": "recover",
            "INIT_ADAPTER_DIR": init_adapter_dir,
        },
    }

def adapter_is_ready(adapter_dir: str) -> bool:
    return (
        os.path.isdir(adapter_dir)
        and os.path.exists(os.path.join(adapter_dir, "adapter_config.json"))
        and (
            os.path.exists(os.path.join(adapter_dir, "adapter_model.safetensors"))
            or os.path.exists(os.path.join(adapter_dir, "adapter_model.bin"))
        )
    )

def build_train_plan_jobs() -> List[Dict[str, Any]]:
    if getattr(CFG, "TRAIN_PLAN", "single") == "curriculum":
        stage_a = build_training_job(
            stage_label="stageA",
            source=CFG.CURRICULUM_STAGEA_SOURCE,
            epochs=float(CFG.CURRICULUM_STAGEA_EPOCHS),
            init_adapter_dir=None,
        )
        stage_b = build_training_job(
            stage_label="stageB",
            source=CFG.CURRICULUM_STAGEB_SOURCE,
            epochs=float(CFG.CURRICULUM_STAGEB_EPOCHS),
            init_adapter_dir=stage_a["adapter_dir"],
        )
        stage_c = build_training_job(
            stage_label="stageC",
            source=CFG.CURRICULUM_STAGEC_SOURCE,
            epochs=float(CFG.CURRICULUM_STAGEC_EPOCHS),
            init_adapter_dir=stage_b["adapter_dir"],
        )
        return [stage_a, stage_b, stage_c]

    return [
        build_training_job(
            stage_label="single",
            source=CFG.LORA_TRAIN_SOURCE,
            epochs=float(CFG.INITIAL_LORA_EPOCH),
            init_adapter_dir=None,
        )
    ]

def run_training_job(job: Dict[str, Any], reuse_existing: bool = True) -> Dict[str, Any]:
    print("=" * 120)
    print("TRAIN JOB :", job["job_name"])
    print("STAGE     :", job["stage_label"])
    print("SOURCE    :", job["source"])
    print("EPOCHS    :", job["epochs"])
    print("ADAPTER   :", job["adapter_dir"])
    print("INIT ADPTR:", job["init_adapter_dir"])

    t0 = time.time()

    if reuse_existing and adapter_is_ready(job["adapter_dir"]):
        elapsed = time.time() - t0
        print("[INFO] Reusing existing adapter.")
        return {
            "job_name": job["job_name"],
            "stage_label": job["stage_label"],
            "source": job["source"],
            "epochs": job["epochs"],
            "adapter_dir": job["adapter_dir"],
            "trainer_output_dir": job["trainer_output_dir"],
            "status": "reused",
            "elapsed_sec": float(elapsed),
            "train_rows": None,
            "valid_rows": None,
        }

    reset_runtime_model_cache(verbose=False)

    with temporary_cfg(**job["overrides"]):
        ensure_dir(CFG.OUTPUT_DIR)
        lora_train_df, lora_valid_df = get_lora_train_valid_split()

        print("LoRA train source:", CFG.LORA_TRAIN_SOURCE)
        print("LoRA train rows  :", len(lora_train_df))
        print("LoRA valid rows  :", 0 if lora_valid_df is None else len(lora_valid_df))
        print("Init adapter dir :", getattr(CFG, "INIT_ADAPTER_DIR", None))
        print("Base model path  :", CFG.BASE_MODEL_PATH)
        print("Adapter dir      :", CFG.ADAPTER_DIR)
        print("Trainer out dir  :", CFG.OUTPUT_DIR)

        local_model, local_tokenizer, local_history_df = train_lora_model(
            lora_train_df,
            lora_valid_df,
            run_name=job["job_name"],
        )

        del local_model
        del local_tokenizer
        del local_history_df

    reset_runtime_model_cache(verbose=False)
    elapsed = time.time() - t0

    return {
        "job_name": job["job_name"],
        "stage_label": job["stage_label"],
        "source": job["source"],
        "epochs": job["epochs"],
        "adapter_dir": job["adapter_dir"],
        "trainer_output_dir": job["trainer_output_dir"],
        "status": "trained",
        "elapsed_sec": float(elapsed),
        "train_rows": int(len(lora_train_df)),
        "valid_rows": 0 if lora_valid_df is None else int(len(lora_valid_df)),
    }

def run_training_jobs_sequential(jobs: List[Dict[str, Any]]) -> pd.DataFrame:
    records = []
    for job in jobs:
        TRAINING_JOB_REGISTRY[job["job_name"]] = job
        record = run_training_job(job, reuse_existing=CFG.REUSE_EXISTING_TRAINED_ADAPTERS)
        records.append(record)
        RUN_TO_TRAIN_JOB[job["job_name"]] = job
        EXPERIMENT_RUN_REGISTRY[job["job_name"]] = {
            "ADAPTER_DIR": job["adapter_dir"],
            "OUTPUT_DIR": job["trainer_output_dir"],
            "LORA_TRAIN_SOURCE": job["source"],
            "INIT_ADAPTER_DIR": job["init_adapter_dir"],
            "TRAIN_PLAN": CFG.TRAIN_PLAN,
            "STAGE_NAME": job["stage_label"],
        }
    result_df = pd.DataFrame(records)
    if len(result_df) > 0:
        result_df = result_df.sort_values(["stage_label", "source", "epochs", "job_name"]).reset_index(drop=True)
    return result_df

TRAIN_PLAN_JOBS = build_train_plan_jobs()
training_results_df = run_training_jobs_sequential(TRAIN_PLAN_JOBS)
display(training_results_df)


TRAIN JOB : single_filtered_seq2048_e1_1p5b
STAGE     : single
SOURCE    : filtered
EPOCHS    : 1.0
ADAPTER   : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b
INIT ADPTR: None
LoRA train source: filtered
LoRA train rows  : 24276
LoRA valid rows  : 2111
Init adapter dir : None
Base model path  : /root/autodl-tmp/models/Qwen/Qwen2.5-Coder-1.5B-Instruct
Adapter dir      : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b
Trainer out dir  : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/trainer_outputs/single_filtered_seq2048_e1_1p5b


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Trainable params: 18,464,768 / 1,562,179,072 (1.1820%)


Step,Training Loss,Validation Loss
250,0.521911,0.517774
500,0.500889,0.504903
750,0.472066,0.498414
1000,0.493877,0.491856
1250,0.470356,0.484482
1500,0.490840,0.481839
1750,0.444526,0.476750
2000,0.450044,0.473463
2250,0.461333,0.469222
2500,0.471883,0.465357


[INFO] Validation checkpoints:


,step,epoch,eval_loss
14,3750,0.617894,0.448440
15,4000,0.659087,0.446766
16,4250,0.700280,0.444622
17,4500,0.741473,0.442440
18,4750,0.782666,0.441208
19,5000,0.823859,0.440081
20,5250,0.865052,0.439208
21,5500,0.906245,0.438556
22,5750,0.947438,0.438291
23,6000,0.988631,0.438298


[INFO] Saved adapter to: /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b
[INFO] Saved trainer history to: /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b/trainer_log_history.csv


,job_name,stage_label,source,epochs,adapter_dir,trainer_output_dir,status,elapsed_sec,train_rows,valid_rows
0,single_filtered_seq2048_e1_1p5b,single,filtered,1.0,/root/autodl-tmp/working/nyu322_pipeline1/nyu3...,/root/autodl-tmp/working/nyu322_pipeline1/nyu3...,trained,10515.168786,24276,2111


In [32]:

# Chunk 16.2 — register final run from the selected train plan

FIXED_HYBRID_OVERRIDES = {
    "SUBMISSION_MODE": "hybrid",
    "DIRECT_RETRIEVAL_THRESHOLD": 1.00,
    "RETRIEVAL_SOURCE": "full",
    "TAG_MODE": "official",
    "SANITIZER_MODE": "recover",
    "MAX_SEQ_LEN": 2048,
    "MAX_NEW_TOKENS": 896,
    "LOAD_EXISTING_ADAPTER": True,
    "DO_TRAIN": False,
}

def build_candidate_run_from_job(job: Dict[str, Any], preset_label: str) -> Dict[str, Any]:
    run_name = f"{preset_label}_{job['job_name']}_hybrid100"
    overrides = {
        **FIXED_HYBRID_OVERRIDES,
        "ADAPTER_DIR": job["adapter_dir"],
    }
    return {
        "run_name": run_name,
        "job_name": job["job_name"],
        "source": job["source"],
        "epochs": job["epochs"],
        "adapter_dir": job["adapter_dir"],
        "overrides": overrides,
    }

FINAL_JOB = TRAIN_PLAN_JOBS[-1]
FINAL_RUN = build_candidate_run_from_job(FINAL_JOB, preset_label=CFG.EXPERIMENT_PRESET)

best_run_name = FINAL_RUN["run_name"]
best_run_overrides = FINAL_RUN["overrides"]

RUN_TO_TRAIN_JOB[best_run_name] = FINAL_JOB["job_name"]
EXPERIMENT_RUN_REGISTRY[best_run_name] = best_run_overrides

quick_experiment_results_df = training_results_df.copy()
quick_experiment_scored = {}
oof_results_df = pd.DataFrame()

FINAL_SUBMISSION_PATH = os.path.join(PIPELINE_SUBMISSION_ROOT, f"submission_{best_run_name}.csv")

print("Best run name         :", best_run_name)
print("Best plan             :", CFG.TRAIN_PLAN)
print("Best source           :", FINAL_JOB["source"])
print("Best epochs           :", FINAL_JOB["epochs"])
print("Best adapter dir      :", FINAL_JOB["adapter_dir"])
print("Final submission path :", FINAL_SUBMISSION_PATH)
print("Final overrides       :", best_run_overrides)


Best run name         : baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100
Best plan             : single
Best source           : filtered
Best epochs           : 1.0
Best adapter dir      : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b
Final submission path : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/submissions/submission_baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100.csv
Final overrides       : {'SUBMISSION_MODE': 'hybrid', 'DIRECT_RETRIEVAL_THRESHOLD': 1.0, 'RETRIEVAL_SOURCE': 'full', 'TAG_MODE': 'official', 'SANITIZER_MODE': 'recover', 'MAX_SEQ_LEN': 2048, 'MAX_NEW_TOKENS': 896, 'LOAD_EXISTING_ADAPTER': True, 'DO_TRAIN': False, 'ADAPTER_DIR': '/root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b'}


In [33]:

# Chunk 16.3 — skipped in direct run version
print("Direct run mode: skip OOF confirm.")


Direct run mode: skip OOF confirm.


In [34]:

# Chunk 16.4 — skipped in direct run version
print("Direct run mode: skip stage2 epoch comparison.")


Direct run mode: skip stage2 epoch comparison.


In [35]:

# Chunk 16.5 — final aliases
best_run_overrides = EXPERIMENT_RUN_REGISTRY.get(best_run_name, {})
FINAL_SUBMISSION_PATH = os.path.join(
    PIPELINE_SUBMISSION_ROOT,
    f"submission_{best_run_name}.csv"
)
print("Forced best_run_name:", best_run_name)
print("Forced adapter dir   :", best_run_overrides.get("ADAPTER_DIR"))


Forced best_run_name: baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100
Forced adapter dir   : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b


In [38]:
# Chunk 17 — preview generation on a few quick-holdout prompts
# 默认用 best_run_name 的配置做预览；如果还没选出来，就用当前 CFG。

# Chunk 17 — preview generation on leak-free quick-holdout prompts

preview_overrides_raw = (
    EXPERIMENT_RUN_REGISTRY.get(best_run_name, {})
    if "best_run_name" in globals() and best_run_name
    else {}
)

_known_cfg_fields = set(getattr(CFG, "__dataclass_fields__", {}).keys())
preview_overrides = {
    k: v for k, v in preview_overrides_raw.items()
    if k in _known_cfg_fields
}

if not getattr(CFG, "ENABLE_PREVIEW_GENERATION", True):
    print("CFG.ENABLE_PREVIEW_GENERATION = False -> skip preview generation")
else:
    if "quick_holdout_split" not in globals():
        raise RuntimeError("quick_holdout_split not found. Please run Chunk 11 first.")

    if "reset_runtime_model_cache" in globals():
        reset_runtime_model_cache(verbose=False)

    with temporary_cfg(**preview_overrides):
        if getattr(CFG, "SUBMISSION_MODE", "hybrid") in {"hybrid", "generation_first"}:
            local_model, local_tokenizer = ensure_inference_model_loaded()
        else:
            local_model, local_tokenizer = (None, None)

        # retrieval bank excludes holdout -> leak-free
        preview_retrieval_index, _ = get_active_valid_retrieval_index_and_bank()

        # eval rows come only from quick holdout
        preview_df = maybe_sample_eval_df(
            quick_holdout_split,
            max_rows=getattr(CFG, "PREVIEW_MAX_ROWS", 24),
            seed=CFG.SEED,
        )

        preview_pred_df, preview_elapsed = run_prediction_pipeline(
            eval_df=preview_df,
            retrieval_index=preview_retrieval_index,
            run_name=f"preview_{best_run_name}" if "best_run_name" in globals() else "preview_run",
            max_rows=None,
            save_debug_csv=False,
        )

    print("preview elapsed sec:", round(preview_elapsed, 2))
    cols_to_show = [
        c for c in [
            "id", "prompt", "retrieval_sim", "route",
            "sanitize_reason", "no_ref_score", "pred_valid"
        ] if c in preview_pred_df.columns
    ]
    display(preview_pred_df[cols_to_show].head(getattr(CFG, "PREVIEW_MAX_ROWS", 24)))

    if "reset_runtime_model_cache" in globals():
        reset_runtime_model_cache(verbose=False)

[INFO] Loading base model from: /root/autodl-tmp/models/Qwen/Qwen2.5-Coder-1.5B-Instruct
[INFO] Loading adapter from: /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

predict:preview_baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100:   0%|          | 0/8 [00:00<?, ?…

preview elapsed sec: 128.6


,id,prompt,retrieval_sim,route,sanitize_reason,pred_valid
0,07cff9b3542c97dd7dc0a9d376869952,The image contains a black icon with a simple ...,0.618064,generated,ok,True
1,faebcf65cd5d338e142d72cf4453858c,A minimalist icon featuring three horizontal l...,0.935414,generated,ok,True
2,2ff4f139fdc412acc2997e37ce88a85a,The image contains a gray television with two ...,0.725454,generated,ok,True
3,7b40e293940bf38cd20e27a46f8fddac,A black square icon with a white mountain silh...,0.641001,fallback,xml_recovered_by_lxml,True
4,a93c002efe466de1c0ebb651469677e0,A simple blue cloud icon with a rounded top an...,0.570645,generated,ok,True
5,d60368febc2863ed30264770326925be,The image features a light blue rectangular sh...,0.593093,generated,ok,True
6,2837396b0191bb4cec3c38bb2d56dbb6,An illustration of a singular black eye shape ...,0.586219,generated,ok,True
7,ddc2f09957f57d3ccfdb6ef67dd7380b,A simple line drawing of a human figure with a...,1.000000,generated,ok,True


In [39]:
# Chunk 18 — unified submission builder
# 支持三种模式：
# 1) retrieval_only
# 2) hybrid
# 3) generation_first
#
# 默认会自动套用 best_run_name 对应的 overrides。
# 同时会额外保存一份 submission_debug.csv，方便你把结果贴给我后我帮你判断。

from tqdm import tqdm

if not CFG.DO_INFERENCE:
    print("CFG.DO_INFERENCE = False -> skip submission build")
else:
    final_overrides_raw = (
        EXPERIMENT_RUN_REGISTRY.get(best_run_name, {})
        if "best_run_name" in globals() and best_run_name
        else {}
    )

    _known_cfg_fields = set(getattr(CFG, "__dataclass_fields__", {}).keys())
    final_overrides = {
        k: v for k, v in final_overrides_raw.items()
        if k in _known_cfg_fields
    }

    if "reset_runtime_model_cache" in globals():
        reset_runtime_model_cache(verbose=False)

    print("Using final run overrides:", final_overrides)

    with temporary_cfg(**final_overrides):
        need_generation = CFG.SUBMISSION_MODE in {"hybrid", "generation_first"}
        local_model, local_tokenizer = (None, None)
        if need_generation:
            local_model, local_tokenizer = ensure_inference_model_loaded()

        submission_retrieval_index, submission_retrieval_bank = get_active_submission_retrieval_index_and_bank()
        preds = []
        debug_rows = []

        for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="submission"):
            prompt = str(row["prompt"])
            nn_svg, sim, nn_idx = submission_retrieval_index.retrieve(prompt)

            pred_info = predict_single_prompt(
                prompt=prompt,
                nn_svg=nn_svg,
                sim=sim,
                model=local_model,
                tokenizer=local_tokenizer,
            )

            preds.append(pred_info["pred_svg"])
            debug_rows.append({
                "id": row["id"],
                "prompt": prompt,
                "retrieval_sim": sim,
                "route": pred_info["route"],
                "sanitize_reason": pred_info["sanitize_reason"],
                "raw_len": pred_info["raw_len"],
                "raw_has_svg": pred_info["raw_has_svg"],
                "raw_has_close": pred_info["raw_has_close"],
                "pred_valid": pred_info["pred_valid"],
            })

        final_submission_path = (
            FINAL_SUBMISSION_PATH
            if "FINAL_SUBMISSION_PATH" in globals()
            else CFG.SUBMISSION_PATH
        )

        fixed_preds = []
        for svg in preds:
            if is_valid_svg(svg, tag_mode=CFG.TAG_MODE) and has_visible_shapes(svg, tag_mode=CFG.TAG_MODE):
                fixed_preds.append(svg)
            else:
                fixed_preds.append(EMPTY_SVG)

        submission = pd.DataFrame({"id": test_df["id"], "svg": fixed_preds})
        submission.to_csv(final_submission_path, index=False)

        debug_df = pd.DataFrame(debug_rows)
        debug_path = os.path.splitext(final_submission_path)[0] + "_debug.csv"
        debug_df.to_csv(debug_path, index=False)

        print("saved submission:", final_submission_path)
        print("saved debug csv :", debug_path)
        print("submission mode :", CFG.SUBMISSION_MODE)
        print("retrieval source:", CFG.RETRIEVAL_SOURCE)
        print("tag mode        :", CFG.TAG_MODE)
        print("sanitizer mode  :", CFG.SANITIZER_MODE)
        print("submission bank size:", len(submission_retrieval_bank))
        print("route counts:", debug_df["route"].value_counts(dropna=False).to_dict())
        print("sanitize reasons:", debug_df["sanitize_reason"].value_counts(dropna=False).head(10).to_dict())
        display(submission.head())
        display(debug_df.head())

if "reset_runtime_model_cache" in globals():
    reset_runtime_model_cache(verbose=False)

Using final run overrides: {'SUBMISSION_MODE': 'hybrid', 'DIRECT_RETRIEVAL_THRESHOLD': 1.0, 'RETRIEVAL_SOURCE': 'full', 'TAG_MODE': 'official', 'SANITIZER_MODE': 'recover', 'MAX_SEQ_LEN': 2048, 'MAX_NEW_TOKENS': 896, 'LOAD_EXISTING_ADAPTER': True, 'DO_TRAIN': False, 'ADAPTER_DIR': '/root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b'}
[INFO] Loading base model from: /root/autodl-tmp/models/Qwen/Qwen2.5-Coder-1.5B-Instruct
[INFO] Loading adapter from: /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/adapters/svg_lora_adapter_single_filtered_seq2048_e1_1p5b


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

submission: 100%|██████████| 1000/1000 [5:31:55<00:00, 19.92s/it] 

saved submission: /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/submissions/submission_baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100.csv
saved debug csv : /root/autodl-tmp/working/nyu322_pipeline1/nyu331_baseline_hybrid100_single_nocanon_v1_pipeline/submissions/submission_baseline_hybrid100_single_filtered_seq2048_e1_1p5b_hybrid100_debug.csv
submission mode : hybrid
retrieval source: full
tag mode        : official
sanitizer mode  : recover
submission bank size: 49999
route counts: {'generated': 818, 'fallback': 147, 'retrieval': 35}
sanitize reasons: {'ok': 597, 'xml_recovered_by_lxml': 368, 'not_called': 35}


,id,svg
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,"<svg fill=""none"" height=""256"" viewBox=""0 0 256..."
1,6eede943219547c22ac56085027d33cc,"<svg viewBox=""0 0 256 256"" height=""256"" width=..."
2,ea045c7a247166f061ce504d9b7ccaab,"<svg viewBox=""0 0 256 256"" height=""256"" width=..."
3,8fe82f3af89e487b31236ca829c3f071,"<svg viewBox=""0 0 256 256"" height=""256"" width=..."
4,600464e4d92c75338462271a09b3f176,"<svg viewBox=""0 0 256 256"" height=""256"" width=..."


,id,prompt,retrieval_sim,route,sanitize_reason,raw_len,raw_has_svg,raw_has_close,pred_valid
0,fa1d8fa7-080f-4269-a9cf-a17562c9a0ca,firewood stack cut logs wood with leaf illustr...,0.468673,generated,ok,1004,True,True,True
1,6eede943219547c22ac56085027d33cc,The image shows five horizontal lines of varyi...,0.788834,generated,xml_recovered_by_lxml,934,True,True,True
2,ea045c7a247166f061ce504d9b7ccaab,A stylized icon depicting a curved arrow withi...,0.907943,generated,ok,736,True,True,True
3,8fe82f3af89e487b31236ca829c3f071,The image contains black geometric shapes agai...,0.613938,generated,ok,935,True,True,True
4,600464e4d92c75338462271a09b3f176,The image shows a single dark gray triangle po...,0.784576,fallback,xml_recovered_by_lxml,125,True,True,True



### Deprecated cell

旧版的 submission 构建代码已被上面的 **Chunk 18 — unified submission builder** 取代。  
现在请只使用这一套统一逻辑：

- `CFG.RETRIEVAL_SOURCE`
- `CFG.SUBMISSION_MODE`
- `CFG.DO_TRAIN`
- `CFG.LOAD_EXISTING_ADAPTER`

这样可以避免：

- `nn_index` 被覆盖后自己没注意到
- 以为在跑 full / filtered，实际跑成 split
- retrieval-only 和 hybrid 逻辑分叉太多


## What changed in this updated direct-run version

This notebook keeps the simplified direct-run path, but adds four important upgrades:

1. **Sequence budgets are fully unified**
   - `MAX_SEQ_LEN = 2048`
   - `MAX_NEW_TOKENS = 896`
   - final run overrides now explicitly keep the 2048 setting

2. **Training-side SVG cleaning is stronger**
   - sanitize first, then compute stats on cleaned SVG
   - keep `svg_raw` for debug
   - mark `visible_ok` and `clean_changed`

3. **Filtered training is now complexity-aware**
   - keeps simple icon / UI style bias
   - additionally uses command count, control-point proxy, command diversity,
     primitive diversity, and long-path features

4. **Single-run training now has validation**
   - a small train/valid split is created from the chosen LoRA source
   - trainer logs are saved next to the adapter
   - still no multi-candidate compare / no OOF model selection


### How to use this NYU322 end-to-end notebook

Run the notebook **top to bottom**.

What it does automatically:

1. Train `filtered @ 2048 @ 1.0`
2. Fix inference to **`hybrid100_official_recover`**
3. Skip OOF
4. Skip stage2
5. Use the single trained adapter as final best run
6. Build submission directly

Important notes:

- This notebook is designed to be **one-click sequential**.
- It saves adapters into dedicated `nyu322_pipeline/adapters/...` directories and reuses them on rerun.
- It clears model objects / CUDA cache between training and inference stages.
- It assumes the final inference route is fixed to **`hybrid100_official_recover`**.
